In [31]:
def virtual_coord_mapping_from_ref_dict(ref_dict: dict, coord_names: list) -> dict:
    '''
    Convert a reference dict to virtual_coord_mapping format.
    
    Parameters:
    -----------
    ref_dict : dict
        Mapping of primary values to tuples of virtual coord values.
        E.g., {'ch0': ('DevA', 'Port0'), 'ch1': ('DevA', 'Port1')}
    
    coord_names : list
        Names for each position in the tuple.
        E.g., ['device', 'localID']
    
    Returns:
    --------
    dict in virtual_coord_mapping format:
        {'ch0': {'device': 'DevA', 'localID': 'Port0'}, ...}
    
    Example:
    --------
    >>> ref_dict = {'ch0': ('DevA', 'P0'), 'ch1': ('DevA', 'P1')}
    >>> mapping = virtual_coord_mapping_from_ref_dict(ref_dict, ['device', 'localID'])
    >>> mapping
    {'ch0': {'device': 'DevA', 'localID': 'P0'}, 'ch1': {'device': 'DevA', 'localID': 'P1'}}
    '''
    return {
        primary: dict(zip(coord_names, values))
        for primary, values in ref_dict.items()
    }

    

In [32]:
# Example usage


ref_dict = {
'ch0': ('DevA', 'Port0'),
'ch1': ('DevA', 'Port1'),
'ch2': ('DevB', 'Port0'),
}
coord_names = ['device', 'localID']
mapping = virtual_coord_mapping_from_ref_dict(ref_dict, coord_names)
print(mapping)

{'ch0': {'device': 'DevA', 'localID': 'Port0'}, 'ch1': {'device': 'DevA', 'localID': 'Port1'}, 'ch2': {'device': 'DevB', 'localID': 'Port0'}}


In [33]:

n_list= []
for n in sorted(ref_dict.keys()):
    n_list.append(n)
    print(n)

  

ch0
ch1
ch2


-----------

In [34]:
################################################################################
# Imports
################################################################################
    
import os
import re
from pathlib import Path
import harp
import pandas as pd
import numpy as np
import xarray as xr
import yaml
from collections.abc import Mapping

################################################################################

def construct_base_da(values: list | np.ndarray | None= None,
                      unified_dim: list = None,
                      unified_coord_name: str= 'unified_coord',
                      virtual_coord_names: list= None,
                      virtual_map_dict: dict= None,
                      dict_of: str= 'dicts',
                      times: list | np.ndarray= None,
                      name: str= 'base_dataarray',
                      test_help_values: bool= False,
                      verbose: bool= False
                      ) -> xr.DataArray:
    '''
    Construct a base/placeholder DataArray for a set of given unified/global coordinates for all timepoints and their associated virtual coordinates.
    -----------------------------------

    Parameters:
        values (list or np.ndarray): Values to populate the DataArray. If None, a placeholder array of NaNs will be created.
        unified_dim (list): List of unified/global coordinate names to include in the DataArray.
        virtual_coord_names (list): List of virtual coordinate dimensions to include in the DataArray.
        virtual_map_dict (dict): Dictionary mapping unified/global coordinates to their virtual coordinate information.
        dict_of (str): Specifies the format of virtual_map_dict. Options are 'dicts' or 'tuples'.
        times (list): List of timepoints to include in the DataArray. This should be obtained using collect_timestamps_dict with return_type='list'.
        name (str): Name of the DataArray.
        test_help_values (bool): If True, use test helper values instead of the main values.
        verbose (bool): If True, print verbose output.
    
    Returns:
        A DataArray populated with the specified values, or a placeholder array if no values are provided.
    '''
    
    from collections.abc import Mapping

    dict_of= dict_of.lower()

    if dict_of not in ("dicts", "tuples"):
        raise ValueError(f"dict_of must be either 'dicts' or 'tuples', got: {dict_of}")

    if times is None:
        raise ValueError("times must be provided.")
    if unified_dim is None:
        raise ValueError("unified_dim must be provided.")
    if virtual_coord_names is None:
        raise ValueError("virtual_coord_names must be provided.")
    if virtual_map_dict is None:
        raise ValueError("virtual_map_dict must be provided.")
    
    missing= [u for u in unified_dim if u not in virtual_map_dict]
    if missing: raise KeyError(f'Error: virtual_map_dict is missing entries for unified coordinates: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    
    if dict_of == "tuples" and any(isinstance(v, dict) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='tuples' but at least one entry is a dict. Use dict_of='dicts'.")

    if dict_of == "dicts" and any(isinstance(v, (tuple, list)) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='dicts' but at least one entry is a tuple/list. Use dict_of='tuples'.")



    #=== Create data to populate DataArray 
    if values is None and test_help_values is False:
        if verbose: print(f'No values provided, test_help_values is False. Creating NaNs placeholder DataArray with {len(times)} timestamps and {len(unified_dim)} unified coordinates.')
        values= np.zeros((len(times), len(unified_dim)), dtype= bool) 

    #=== Create test helper values if specified
    if values is None and test_help_values is True:
        if verbose: print(f'No values provided, but test_help_values is True. Creating test helper DataArray with {len(times)} timestamps and {len(unified_dim)} unified coordinates.')
        value_list = []
        for _ in times:
            time_values = []
            for unified in unified_dim:
                virtual_coords = virtual_map_dict[unified]

                if dict_of == 'dicts':
                    if not isinstance(virtual_coords, Mapping): raise TypeError(f'Expected virtual_map_dict to be dict-of-dicts when dict_of="dicts", but got {type(virtual_coords)} for unified coordinate {unified}')
                    
                    coord_str = ', '.join([f'{k}: {v}' for k, v in virtual_coords.items()])

                elif dict_of == 'tuples':
                    
                    if not isinstance(virtual_coords, (tuple, list)): raise TypeError(f'Expected virtual_map_dict to be dict-of-tuples/lists when dict_of="tuples", but got {type(virtual_coords)} for unified coordinate {unified}')
                    
                    if len(virtual_coords) != len(virtual_coord_names): raise ValueError(f"Tuple length mismatch for {unified}: len(entry)={len(virtual_coords)} vs len(virtual_coord_names)={len(virtual_coord_names)}")
                    
                    coord_str = ", ".join([f"{k}: {v}" for k, v in zip(virtual_coord_names, virtual_coords)])

                else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

                time_values.append(f'({unified}, {coord_str})')

            value_list.append(time_values)
        values = np.array(value_list)
        if verbose: print(f'Constructed test helper values array with shape: {values.shape}')
    
    if values is not None and test_help_values is True:
        print('Warning: Both values provided and test_help_values is True. Using provided values.')
    
    #=== Build coordinates 
    
    # Primary/Base coordinates
    coords= {
        'Time': times,
        unified_coord_name: unified_dim,
        }
    
    # Virtual coordinates (dict-of-dicts / named fields)
    if dict_of == 'dicts':
        for vcoord in virtual_coord_names:
            try: vcoord_values= [virtual_map_dict[unified][vcoord] for unified in unified_dim]
            except KeyError as e:
                raise KeyError(f'Missing key {e} in virtual_map_dict entries (dict_of="dicts")') from e
            
            coords[vcoord] = (unified_coord_name, vcoord_values)
    
    
    # Virtual coordinates (dict-of-tuples / positional fields)
    elif dict_of == 'tuples':
        for i, vcoord in enumerate(virtual_coord_names):
            try: vcoord_values = [virtual_map_dict[unified][i] for unified in unified_dim]
            except IndexError as e:
                raise IndexError(f'Index {i} out of range for virtual_map_dict entries (dict_of="tuples")') from e
            
            coords[vcoord] = (unified_coord_name, vcoord_values)
    
    else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

    ########################################################
    """
    NOTE: virtual_map_dict can be either:
      (A) dict-of-dicts (recommended):
          virtual_map_dict[unified] = {"device": ..., "localID": ..., ...}
          -> use key lookup: virtual_map_dict[unified][vcoord]
    
      (B) dict-of-tuples/lists (positional):
          virtual_map_dict[unified] = (device, localID, ...)
          -> assumes tuple order matches virtual_coord_names
    
    Example for (B):
    for i, vcoord in enumerate(virtual_coord_names):
        vcoord_values = [virtual_map_dict[unified][i] for unified in unified_dim]
        coords[vcoord] = (unified_coord_name, vcoord_values)
    
    Dict-of-tuples requires that the order of elements in the tuple matches the order in virtual_coord_names.
    """
    ########################################################

    #=== Create DataArray
    da= xr.DataArray(
        data=values,
        dims= ['Time', unified_coord_name],
        coords= coords,
        name= name,
        attrs= {
            'description': f'Base DataArray for unified coordinates of type {name}',
            'source': 'Constructed using construct_base_da function'
        }
    )
    return da


###########################

experiment_directory_path= './Bonsai_logs/2025-10-07T18-08-45'
harp_device_yaml_path= './device.yml'

device_list= ['Behavior0',
               'Behavior1',
               'Behavior2',
               'Behavior3',
               'Behavior4',
               'Behavior5',
]

device_IDs= {
    f'{device_list[0]}': 'ID_0',
    f'{device_list[1]}': 'ID_1',
    f'{device_list[2]}': 'ID_2',
    f'{device_list[3]}': 'ID_3',
    f'{device_list[4]}': 'ID_4',
    f'{device_list[5]}': 'ID_5',
}

device_registers_dict= {
    'Behavior0': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior1': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior2': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior3': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior4': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior5': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
}

Nosepoke_Activations_localIDs= ['DIPort0',
                                'DIPort1',
                                'DIPort2',
]

Nosepoke_LED_Activations_localIDs= ['DOPort0',
                                    'DOPort1',
                                    'DOPort2',
]

Nosepoke_Valve_Activations_localIDs= ['SupplyPort0',
                                      'SupplyPort1',
                                      'SupplyPort2',
]

Nosepoke_Reward_Release_localIDs= ['PulseSupplyPort0',
                                   'PulseSupplyPort1',
                                   'PulseSupplyPort2',
]

channel_list= ['Nosepoke0',
               'Nosepoke1',
               'Nosepoke2',
               'Nosepoke3',
               'Nosepoke4',
               'Nosepoke5',
               'Nosepoke6',
               'Nosepoke7',
               'Nosepoke8',
               'Nosepoke9',
               'Nosepoke10',
               'Nosepoke11',
               'Nosepoke12',
               'Nosepoke13',
               'Nosepoke14',
               'Nosepoke15',
               'Nosepoke16',
               'Nosepoke17',]


channel_type_localIDs= {
    'Activations': Nosepoke_Activations_localIDs,
    'LED_Activations': Nosepoke_LED_Activations_localIDs,
    'Valve_Activations': Nosepoke_Valve_Activations_localIDs,
    'Reward_Release': Nosepoke_Reward_Release_localIDs,
}

Activations_localID_register_dict= {
    'DIPort0': '32',
    'DIPort1': '32',
    'DIPort2': '32',
}
LED_Activations_localID_register_dict= {
    'DOPort0': '34',
    'DOPort1': '34',
    'DOPort2': '34',
}

Valve_Activations_localID_register_dict= {
    'SupplyPort0': '34',
    'SupplyPort1': '34',
    'SupplyPort2': '34',
}

Reward_Release_localID_register_dict= {
    'PulseSupplyPort0': '49',
    'PulseSupplyPort1': '50',
    'PulseSupplyPort2': '51',
}

channel_type_registerIDs= {
    'Activations': Activations_localID_register_dict,
    'LED_Activations': LED_Activations_localID_register_dict,
    'Valve_Activations': Valve_Activations_localID_register_dict,
    'Reward_Release': Reward_Release_localID_register_dict,
}

type_key= list(channel_type_localIDs.keys())[0]  # Type of channel to process (e.g., 'Activations'). Must be exact prefix from channel_type_localIDs keys

channel_ref_dict= {
    f'{channel_list[0]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[1]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[2]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[3]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[4]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[5]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[6]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[7]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[8]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[9]}':  (f'{device_list[3]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[10]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[11]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[12]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[13]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[14]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[15]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[16]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[17]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][2]}'),
}

# channel_ref_dict= {
#                 f'{channel_list[0]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[1]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[2]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[3]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[4]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[5]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[6]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[7]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[8]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[9]}':  {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[10]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[11]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[12]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[13]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[14]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[15]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[16]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[17]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},
#             }
# channel_dict= {channel: (device, localID) for channel, (device, localID) in channel_ref_dict.items()}


device_type= 'Behavior'  # Type of device to process (e.g., 'Behavior'). Must be exact prefix 

channel_type_da_name= f'{type_key}'
print(f'Processing channel type: {channel_type_da_name}')

###########################################


def construct_lookup_array(unified_dim: list= None,
                           unified_coord_name: str= 'unified_coord',
                           virtual_coord_names: list= None,
                           virtual_map_dict: dict= None,
                           dict_of: str= 'dicts',
                           name: str= 'lookup_matrix',
                           verbose: bool= False
                           ) -> tuple[xr.DataArray, dict[str, list]]:
    '''
    Construct a Lookup Array DataArray for a set of given unified/global coordinates and their associated virtual coordinates.
    -----------------------------------

    Parameters:
        unified_dim (list): List of unified/global coordinate names to include in the DataArray.
        unified_coord_name (str): Name for the unified/global coordinate dimension.
        virtual_coord_names (list): List of virtual coordinate dimensions to include in the DataArray.
        virtual_map_dict (dict): Dictionary mapping unified/global coordinates to their virtual coordinate information.
        dict_of (str): Specifies the format of virtual_map_dict. Options are 'dicts' or 'tuples'.
        name (str): Name of the DataArray.
        verbose (bool): If True, prints additional information. 
    
    Returns:
        lookup_da (xr.DataArray): A DataArray representing the lookup matrix.
        lookup_virtual_coords (dict): A dictionary containing lists of unique virtual coordinate values for each virtual coordinate dimension. Keys are virtual coordinate names, values are lists of unique values. 
    '''
    #==== Input validation and preprocessing ====
    dict_of= dict_of.lower()

    if dict_of not in ("dicts", "tuples"):
        raise ValueError(f"dict_of must be either 'dicts' or 'tuples', got: {dict_of}")
    
    if unified_dim is None or virtual_coord_names is None or virtual_map_dict is None:
        raise ValueError("unified_dim, virtual_coord_names, and virtual_map_dict must be provided.")
    
    if len(unified_dim) == 0:
        raise ValueError("unified_dim cannot be empty.")
    if len(virtual_coord_names) == 0:
        raise ValueError("virtual_coord_names cannot be empty.")
    
    if verbose: print(f'Constructing lookup array for {len(unified_dim)} unified coordinates and virtual coordinates: {virtual_coord_names}')
    
    missing= [u for u in unified_dim if u not in virtual_map_dict]
    if missing: raise KeyError(f'Error: virtual_map_dict is missing entries for unified coordinates: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    
    if dict_of == "tuples" and any(isinstance(v, dict) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='tuples' but at least one entry is a dict. Use dict_of='dicts'.")

    if dict_of == "dicts" and any(isinstance(v, (tuple, list)) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='dicts' but at least one entry is a tuple/list. Use dict_of='tuples'.")

    #==== Collect Unique Values for Each Virtual Coordinate (Levels)
    lookup_virtual_coords= {vcoord: [] for vcoord in virtual_coord_names}

    for unified in unified_dim:
        virtual_coords= virtual_map_dict[unified]

        if dict_of == 'tuples':
            if not isinstance(virtual_coords, (tuple, list)):
                raise TypeError(f'Expected virtual_map_dict to be dict-of-tuples/lists when dict_of="tuples", but got {type(virtual_coords)} for unified coordinate {unified}')
            if len(virtual_coords) != len(virtual_coord_names):
                raise ValueError(f"Tuple length mismatch for {unified}: len(entry)={len(virtual_coords)} vs len(virtual_coord_names)={len(virtual_coord_names)}")
            
            for vcoord_name, vcoord_value in zip(virtual_coord_names, virtual_coords):
                if vcoord_value not in lookup_virtual_coords[vcoord_name]:
                    lookup_virtual_coords[vcoord_name].append(vcoord_value)

        elif dict_of == 'dicts':
            if not isinstance(virtual_coords, Mapping):
                raise TypeError(f'Expected virtual_map_dict to be dict-of-dicts when dict_of="dicts", but got {type(virtual_coords)} for unified coordinate {unified}')
            
            for vcoord_name in virtual_coord_names:
                if vcoord_name not in virtual_coords:
                    raise KeyError(f'Missing key {vcoord_name} in virtual_map_dict entry for unified coordinate {unified} (dict_of="dicts")')
                
                vcoord_value= virtual_coords[vcoord_name]
                if vcoord_value not in lookup_virtual_coords[vcoord_name]:
                    lookup_virtual_coords[vcoord_name].append(vcoord_value)
        
        else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

        if verbose: 
            for vcoord_name in virtual_coord_names: 
                print(f'Collected {vcoord_name}: {lookup_virtual_coords[vcoord_name]} unique values.')
        

    #==== Construct Lookup Array ====
    
    # Fast lookup tables from virtual coordinate values to integer indices along each virtual dimension
    index_map= {
        vcoord: {val: idx for idx, val in enumerate(lookup_virtual_coords[vcoord])} 
        for vcoord in virtual_coord_names
        }

    # Allocate boolean array for lookup matrix
    lookup_shape= [len(unified_dim)] + [len(lookup_virtual_coords[vcoord]) for vcoord in virtual_coord_names]
    lookup_arr= np.zeros(lookup_shape, dtype= bool)

    # Populate lookup array
    for i, unified in enumerate(unified_dim):
        
        entry= virtual_map_dict[unified]

        if dict_of == 'tuples':
            idx= [i] + [index_map[vcoord_name][vcoord_value] 
            for vcoord_name, vcoord_value in zip(virtual_coord_names, entry)
            ]
        
        elif dict_of == 'dicts':
            idx= [i] + [index_map[vcoord_name][entry[vcoord_name]]
            for vcoord_name in virtual_coord_names
            ]
        else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")
        lookup_arr[tuple(idx)] = True
    
    #==== Create DataArray 
    dims= [unified_coord_name] + virtual_coord_names
    
    coords= {unified_coord_name: unified_dim,}

    coords.update({vcoord: lookup_virtual_coords[vcoord] for vcoord in virtual_coord_names})

    lookup_da= xr.DataArray(
        data= lookup_arr,
        dims= dims,
        coords= coords,
        name= name,
        attrs= {
            'description': 'Lookup matrix indicating presence of unified coordinates across virtual coordinate dimensions',
            'source': 'Constructed using construct_lookup_array function'
        }
    )
    return lookup_da, lookup_virtual_coords

################################################
def ulookup(lookup_arr: xr.DataArray = None,
            unified_coord_name: str= 'unified_coord',
            verbose: bool= False,
            **selectors,
        ) -> list:
    '''
    uLookup: Selects unified/global coordinates from a lookup DataArray based on specified criteria across virtual coordinate dimensions. Supports both positional- and label-based indexing and combinations thereof.
    -----------------------------------

    Parameters:
        lookup_arr (xr.DataArray): N-dim Boolean lookup tensor with dimensions like [unified_coord_name] + virtual coordinate dimensions.
        unified_coord_name (str): Name of the unified/global coordinate dimension.
        verbose (bool): If True, prints additional information.
        **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by. 
    
    Returns:
        list: A list of unified/global coordinate names that match the specified criteria.
    '''
    if lookup_arr is None:
        raise ValueError("lookup_arr must be provided.")
    if unified_coord_name not in lookup_arr.dims:
        raise ValueError(f"{unified_coord_name} is not a dimension in the provided lookup_arr.")
    if verbose: print(f'Starting uLookup with unified_coord_name="{unified_coord_name}" and selectors: {selectors}')


    #=== Gather selection criteria
    sel= {}

    sel= {key: value for key, value in selectors.items() if value is not None}
    if verbose: print(f'Selecting unified coordinates with criteria: {sel}')

    #=== Get subset based on selection criteria
    subset= lookup_arr.sel(**sel) if sel else lookup_arr
    if verbose: print(f'Subset shape after selection: {subset.shape}')

    #=== Reduce across non-unified_coord_name dimensions
    other_dims= [d for d in subset.dims if d != unified_coord_name]
    
    if other_dims:
        per_unified_true= subset.any(dim= other_dims)
    else:
        per_unified_true= subset.astype(bool)
    
    if verbose: print(f'Per-unified_coord_name true shape after reduction: {per_unified_true.shape}')
    
    # return per_unified_true[unified_coord_name].values[per_unified_true.values].tolist()

    labels = per_unified_true[unified_coord_name].values
    mask = np.asarray(per_unified_true.values, dtype=bool)
    return labels[mask].tolist()
################################################

@xr.register_dataarray_accessor('ul')
class LookupAccessorConstructor:
    '''
    Base class for making custom xarray DataArray accessors for lookup operations with auto-construction of lookup arrays.
    '''

    def __init__(self,
                 data_array: xr.DataArray
                 ):
        
        self._da= data_array

        lookup_array= self.lookup_constructor(lookup_array= None,
                                             unified_coord_name= None,
                                             dict_of= None,
                                             verbose= False,
                                             )
        self.lookup_array= lookup_array



    def lookup_constructor(self,
                           lookup_array: xr.DataArray | None= None,
                           unified_coord_name: str | None = None,
                           dict_of: str | None= None,
                           verbose: bool= False,
                           ):
        
        if lookup_array: 
            self.lookup_array= lookup_array
            if verbose: print(f'Using provided lookup_array with shape {self.lookup_array.shape}.')

        if dict_of is None:
            if verbose: print('No preferred format for "dict_of" provided, defaulting to "tuples".')
            dict_of= 'tuples'
        elif dict_of.lower() not in ('tuples', 'dicts'):
            raise ValueError(f'Invalid dict_of value: {dict_of}. Must be either "tuples" or "dicts".')
        
        if not hasattr(self, 'lookup_array'):
            
            if verbose: print(f'Lookup array not provided, attempting to auto-construct using data array')

            #==== 1) Unified Coordinate Name
            """
            Name of unified/global coordinate dimension in the lookup array. Assumed to be second dimension in the data array, or first dimension not including 'Time' dimension.
            """
            if unified_coord_name is None:
                unified_coord_name= [dim for dim in self._da.dims if dim.lower() != 'time'][0]
                if verbose: print(f'No unified_coord_name provided, inferred as "{unified_coord_name}" from data array dimensions.')
            
            #==== 2) Unified Dimension Values
            """
            List of unified/global coordinate names to include in the DataArray.
            """
            unified_dim = self._da.coords[unified_coord_name].values.tolist()
            if verbose: print(f'Extracted {len(unified_dim)} unified coordinates from data array dimension "{unified_coord_name}".')

            #==== 3) Virtual Coordinate Names
            """
            Names of coordinates that map each unified/global coordinate to its virtual coordinate information.
            Assumed to be all coordinates in the data array except for the unified coordinate and 'Time' dimension.
            """
            virtual_coord_names= [coord for coord in self._da.coords.keys() if coord != unified_coord_name and coord.lower() != 'time']
            if verbose: print(f'Inferred virtual coordinate names: {virtual_coord_names}.')

            #==== 4) Virtual Map Dictionary
            """
            Dictionary mapping unified/global coordinates to their virtual coordinate information. Can be in dict-of-dicts or dict-of-tuples format.
            """
            if dict_of == 'tuples':
                # Fix: Use dict() to create proper dictionary from zip
                virtual_map_dict = dict(zip(
                    unified_dim, 
                    zip(*(self._da.coords[coord].values.tolist() for coord in virtual_coord_names))
                ))
            elif dict_of == 'dicts':
                virtual_map_dict = {
                    unified: {vcoord: self._da.sel({unified_coord_name: unified})[vcoord].item() for vcoord in virtual_coord_names}
                    for unified in unified_dim
                }
            if verbose: print(f'Constructed virtual_map_dict with {len(virtual_map_dict)} entries in format "{dict_of}".')

            #==== 5) Construct Lookup Array
            self.lookup_array, _ = construct_lookup_array(
                unified_dim= unified_dim,
                unified_coord_name= unified_coord_name,
                virtual_coord_names= virtual_coord_names,
                virtual_map_dict= virtual_map_dict,
                dict_of= dict_of,
                name= f'{unified_coord_name}_lookup_matrix',
                verbose= verbose
            )
            if verbose: print(f'Constructed lookup_array with shape {self.lookup_array.shape}.')
        return self.lookup_array
    
    def ul_sel(self,
                verbose: bool= False,
                **selectors,
            ) -> list:
        '''
        uLookup: Selects unified/global coordinates from the lookup DataArray based on specified criteria across virtual coordinate dimensions. Supports both positional- and label-based indexing and combinations thereof.
        -----------------------------------

        Parameters:
            verbose (bool): If True, prints additional information.
            **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by.
        Returns:
            list: A list of unified/global coordinate names that match the specified criteria.
        '''
        u_c_n= [dim for dim in self._da.dims if dim.lower() != 'time'][0]
        return ulookup(
            lookup_arr= self.lookup_array,
            unified_coord_name= u_c_n,
            verbose= verbose,
            **selectors
        )
    
    def sel(self, **selectors) -> list:
        '''
        sel: Alias for ulookup method to select unified/global coordinates based on specified criteria.
        -----------------------------------

        Parameters:
            **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by.
        Returns:
            list: A list of unified/global coordinate names that match the specified criteria.
        '''
        return self.ul_sel(**selectors)
    
    __call__= sel


######################################
from Refactor.HarpExtender.harp_extender_core import collect_device_dfs

experiment_directory_path= './Bonsai_logs/2025-10-07T18-08-45'
harp_device_yaml_path= './device.yml'


device_dfs_dict= collect_device_dfs(
    experiment_directory_path= experiment_directory_path,
    harp_device_yaml_path= harp_device_yaml_path,
    device_type= device_type,
    device_list= device_list,
    device_IDs= device_IDs,
    device_registers_dict= device_registers_dict,
    flatten= False,
    verbose= False,
    validate= False,
)

display(device_dfs_dict)

############################################
from Refactor.Timestamps.timestamps import collect_timestamps_nested_dict

times= collect_timestamps_nested_dict(
    dfs_dict= device_dfs_dict,
    data_key= type_key,
    peripheral_registerIDs= channel_type_registerIDs,
    devices= device_list,
    return_type= 'list',
    verbose= False
)


print(len(times))
# test construct_channel_type_da
test_da= construct_base_da(
    unified_dim= channel_list,
    unified_coord_name= 'channel',
    virtual_coord_names= ['device', 'localID'],
    virtual_map_dict= channel_ref_dict,
    dict_of= 'tuples',
    times= times,
    name= f'{type_key}_data_array',
    verbose= False

)
print(len(times))

display(test_da.to_dataframe())
#############################################


Processing channel type: Activations


/tmp/ipykernel_2013889/4227196408.py:533: AccessorRegistrationWarning: registration of accessor <class '__main__.LookupAccessorConstructor'> under name 'ul' for type <class 'xarray.core.dataarray.DataArray'> is overriding a preexisting attribute with the same name.
  @xr.register_dataarray_accessor('ul')


{'Behavior0': {'8':           TimestampSeconds
  Time                      
  104986.0            104986
  104987.0            104987
  104988.0            104988
  104989.0            104989
  104990.0            104990
  ...                    ...
  106294.0            106294
  106295.0            106295
  106296.0            106296
  106297.0            106297
  106298.0            106298
  
  [1313 rows x 1 columns],
  '32':                DIPort0  DIPort1  DIPort2    DI3
  Time                                           
  105633.255072     True    False    False  False
  105633.568576    False    False    False  False
  105673.389504     True    False    False  False
  105673.471680    False    False    False  False
  106035.302304    False     True    False  False
  106035.322752    False    False    False  False
  106043.345664    False    False     True  False
  106043.443744    False    False    False  False,
  '34':                DOPort0  DOPort1  DOPort2  SupplyPort0  Suppl

54
54


device  localID  Activations_data_array
Time          channel                                               
105136.375456 Nosepoke0   Behavior0  DIPort0                   False
              Nosepoke1   Behavior0  DIPort1                   False
              Nosepoke2   Behavior0  DIPort2                   False
              Nosepoke3   Behavior1  DIPort0                   False
              Nosepoke4   Behavior1  DIPort1                   False
...                             ...      ...                     ...
106094.554592 Nosepoke13  Behavior4  DIPort1                   False
              Nosepoke14  Behavior4  DIPort2                   False
              Nosepoke15  Behavior5  DIPort0                   False
              Nosepoke16  Behavior5  DIPort1                   False
              Nosepoke17  Behavior5  DIPort2                   False

[972 rows x 3 columns]

In [35]:
from Refactor.LookupArrays.Lookup_Arrays_mod import construct_data_array

channel_type_da= construct_data_array(virtual_map= channel_ref_dict,
                                      global_coord_name= 'channel',
                                      virtual_coord_names= ['device', 'localID'],
                                      dict_of= 'tuples',
                                      times= times,
                                      name= f'{type_key}_data_array',
                                      test_values= True,
                                      verbose= True
                                        )


display(channel_type_da.to_dataframe().unstack('channel').drop(columns=['device', 'localID']))


No da_attrs provided, using defaults.
No values provided, test_values is True. Creating test values DataArray with 54 timestamps and 18 unified coordinates.
Constructed test helper values array with shape: (54, 18)
Constructed DataArray "Activations_data_array" with dimensions: ('Time', 'channel') and shape: (54, 18)


Activations_data_array  \
channel                                               Nosepoke0   
Time                                                              
105136.375456  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105136.670336  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105164.753920  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105164.971648  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105451.754080  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105451.893088  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105539.656512  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105539.954304  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.648256  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.684160  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.701728  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.788480  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105633.255072  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105633.568576  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105673.389504  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105673.471680  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105697.369472  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105697.573664  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105766.290784  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105766.619072  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105782.499168  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105782.553984  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105786.995264  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105787.055936  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.470816  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.591968  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.608864  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.743616  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105792.610464  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105792.791680  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105798.357952  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105798.541792  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105799.842304  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105799.906944  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.543296  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.566944  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.584288  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.825312  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105860.990208  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105861.171392  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105883.071616  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105883.147104  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105908.860672  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105909.231936  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105924.389280  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105924.718752  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105935.863360  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105935.956160  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106035.302304  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106035.322752  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106043.345664  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106043.443744  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106094.423488  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106094.554592  (Nosepoke0, device: Behavior0, localID: DIPort0)   

                                                                 \
channel                                               Nosepoke1   
Time                                                              
105136.375456  (Nose

In [36]:
from codecs import lookup
from Refactor.LookupArrays.Lookup_Arrays_mod import construct_lookup_array

lookup_da, lookup_virtual_coords= construct_lookup_array(virtual_map= channel_ref_dict,
                                                         global_coord_name= 'channel',
                                                         virtual_coord_names= ['device', 'localID'],
                                                            dict_of= 'tuples',


)

display(lookup_da.to_dataframe())

,device,localID,lookup_matrix
channel,,,
Nosepoke0,Behavior0,DIPort0,True
Nosepoke1,Behavior0,DIPort1,True
Nosepoke2,Behavior0,DIPort2,True
Nosepoke3,Behavior1,DIPort0,True
Nosepoke4,Behavior1,DIPort1,True
Nosepoke5,Behavior1,DIPort2,True
Nosepoke6,Behavior2,DIPort0,True
Nosepoke7,Behavior2,DIPort1,True
Nosepoke8,Behavior2,DIPort2,True


In [37]:
from Refactor.LookupArrays.Lookup_Arrays_mod import ulookup

# selection= ulookup(lookup_arr= lookup_da,
#                    global_coord_name= 'channel',
#                    verbose= True,

# )
################################################################################
# Comprehensive Test Suite
################################################################################

def test_ulookup_comprehensive():
    """Run all test cases from your test_da_indexing function"""
    
    # Build lookup
    lookup_da, virtual_coords = construct_lookup_array(
        virtual_map=channel_ref_dict,
        global_coord_name='channel',
        virtual_coord_names=['device', 'localID'],
        dict_of='tuples'
    )
    
    devices = virtual_coords['device']
    localIDs = virtual_coords['localID']
    channels = list(channel_ref_dict.keys())
    
    test_cases = [
        ("A", "Slice devices by label range, and take the 2nd localID", 
         {"device": slice('Behavior1','Behavior5'), "localID": 'DIPort1'}),
        
        ("B", "Slice localIDs by label range; any device", 
         {"localID": slice('DIPort0','DIPort2')}),
        
        ("C", "List of devices; list of localIDs", 
         {"device": ['Behavior0','Behavior2'], "localID": ['DIPort0','DIPort1']}),
        
        ("D", "Slice channels by label range", 
         {"channel": slice('Nosepoke2','Nosepoke6')}),
        
        ("E", "Exact single cell", 
         {"channel": 'Nosepoke4', "device": 'Behavior1', "localID": 'DIPort1'}),
        
        ("F", "All channels (wildcard)", 
         {"channel": None, "device": None, "localID": None}),
        
        ("G", "Slice using numeric index for channels", 
         {"channel": channels[0:2]}),
        
        ("H", "Slice using numeric index for devices", 
         {"device": devices[0:2]}),
        
        ("I", "Slice using numeric index for localIDs", 
         {"localID": localIDs[0:2]}),
        
        ("J", "Slice using label range for devices and localIDs", 
         {"device": slice('Behavior1', 'Behavior2'), "localID": slice('DIPort0', 'DIPort2')}),
        
        ("K", "Slice using numeric index for devices and localIDs", 
         {"device": devices[1:3], "localID": localIDs[0:3]}),
        
        ("L", "Slice using label range for channels", 
         {"channel": slice('Nosepoke2', 'Nosepoke6'), "device": None, "localID": None}),
        
        ("M", "Slice using numeric index for channels", 
         {"channel": channels[2:7], "device": devices[:], "localID": localIDs[:]})
    ]
    
    results = {}
    
    for test_id, description, params in test_cases:
        print(f"\n{test_id}) {description}")
        print(f"   Params: {params}")
        
        try:
            result = ulookup(lookup_da, 'channel', **params)
            results[test_id] = {"success": True, "result": result}
            print(f"   ✅ Result: {result}")
        except Exception as e:
            results[test_id] = {"success": False, "error": str(e)}
            print(f"   ❌ ERROR: {e}")
    
    # Summary
    success_count = sum(1 for r in results.values() if r["success"])
    total_count = len(results)
    
    print("\n" + "="*80)
    print(f"TEST SUMMARY: {success_count}/{total_count} passed")
    print("="*80)
    
    if success_count == total_count:
        print("🎉 ALL TESTS PASSED!")
    else:
        print("❌ Some tests failed:")
        for test_id, result in results.items():
            if not result["success"]:
                print(f"  {test_id}: {result['error']}")
    
    return results

# Run tests
if __name__ == "__main__":
    results = test_ulookup_comprehensive()


A) Slice devices by label range, and take the 2nd localID
   Params: {'device': slice('Behavior1', 'Behavior5', None), 'localID': 'DIPort1'}
   ✅ Result: ['Nosepoke4', 'Nosepoke7', 'Nosepoke10', 'Nosepoke13', 'Nosepoke16']

B) Slice localIDs by label range; any device
   Params: {'localID': slice('DIPort0', 'DIPort2', None)}
   ✅ Result: ['Nosepoke0', 'Nosepoke1', 'Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5', 'Nosepoke6', 'Nosepoke7', 'Nosepoke8', 'Nosepoke9', 'Nosepoke10', 'Nosepoke11', 'Nosepoke12', 'Nosepoke13', 'Nosepoke14', 'Nosepoke15', 'Nosepoke16', 'Nosepoke17']

C) List of devices; list of localIDs
   Params: {'device': ['Behavior0', 'Behavior2'], 'localID': ['DIPort0', 'DIPort1']}
   ✅ Result: ['Nosepoke0', 'Nosepoke1', 'Nosepoke6', 'Nosepoke7']

D) Slice channels by label range
   Params: {'channel': slice('Nosepoke2', 'Nosepoke6', None)}
   ✅ Result: ['Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5', 'Nosepoke6']

E) Exact single cell
   Params: {'channel': 'Nosepo

In [38]:
# from Refactor.LookupArrays.Lookup_Arrays_mod import LookupAccessorConstructor

# Build DataArray
da = construct_data_array(
    virtual_map=channel_ref_dict,
    global_coord_name='channel',
    virtual_coord_names=['device', 'localID'],
    times=times,
    dict_of='tuples'
)

# Use accessor (lookup built in __init__)
channels = da.ulookup(device='Behavior1')
# → ['Nosepoke3', 'Nosepoke4', 'Nosepoke5']

# Alternative syntax
channels = da.ulookup.sel(device='Behavior1')
# → Same result

# With verbose
channels = da.ulookup(verbose=True, device='Behavior1')

# Complex queries
channels = da.ulookup(
    device=slice('Behavior1', 'Behavior3'),
    localID='DIPort0'
)

# All channels
channels = da.ulookup()  # No selectors = all channels

Selecting from lookup DataArray with dimensions: ('channel',) and shape: (18,)
Selectors provided: {'device': 'Behavior1'}
Applying following selectors: {'device': 'Behavior1'}
Single value selector on device: Behavior1, resulting shape: (18,)
After applying selector on device, resulting shape: (3,)
Selected 3 channel(s)
  Result: ['Nosepoke3', 'Nosepoke4', 'Nosepoke5']


In [39]:

def update_channel_type_da(
        channel_type_da= None, # Existing DataArray to update
        channel_ref_dict= None,     # {channel: (device, localID)}
        dict_of_devices_registers_dfs_dicts= None, # {device: {register_address: DataFrame(index= Time, columns= localIDs)}}
        device_register_dfs_dict= None, # {device: DataFrame(index= Time, columns= localIDs)}
        channel_type_registerIDs= None, # {type_key: {localID: register_address}}
        type_key= None,
        fill_value= None,
        verbose= False

):
    
    '''
    Update channel_type_da at (Time, channel) positions using: dict_of_devices_registers_dfs_dicts[device][reg_address][localID].   

    Steps per channel:
        Map channel -> (device, localID) via channel_ref_dict
        Map localID -> reg_address via channel_type_registerIDs[type_key][localID]
        Pull source column; align Time to channel_type_da.Time (intersection)
        Optional fill within aligned rows; drop NaNs
        Collect columns into a (Time x channel) patch and assign once

    Parameters:
        channel_type_da (xr.DataArray): Existing DataArray to update.
        channel_ref_dict (dict): Dictionary mapping channel names to their reference information.
        dict_of_devices_registers_dfs_dicts (dict): Nested dictionary of devices and their register DataFrames.
        device_register_dfs_dict (dict or None): Optional pre-constructed dict of device DataFrames to use instead of nested dict.
        channel_type_registerIDs (dict): Mapping of type_key to localID to register_address.
        type_key (str): The type_key corresponding to the channel_type_da.
        fill_value: Value to use for missing data. If None, will use NaN for floats and False for bools.
        verbose (bool): If True, prints detailed output during the update process.

    Returns:
        xr.DataArray: Updated DataArray with new data.
    '''

    #===== Error Checks =====
    if dict_of_devices_registers_dfs_dicts is None:
        raise ValueError("dict_of_devices_registers_dfs_dicts is required for this update path.")
    if channel_type_registerIDs is None or type_key is None:
        raise ValueError("channel_type_registerIDs and type_key are required.")
    if type_key not in channel_type_registerIDs:
        raise ValueError(f"type_key '{type_key}' not found in channel_type_registerIDs.")
    if channel_type_da is None:
        raise ValueError("channel_type_da is required.")
    if channel_ref_dict is None:
        raise ValueError("channel_ref_dict is required.")
    if not isinstance(channel_type_da, xr.DataArray):
        raise ValueError("channel_type_da must be an xarray DataArray.")
    

    #===== Setup =====

    ctda= channel_type_da
    ctda_time= ctda.get_index('Time')
    ctda_channels= ctda.coords['channel'].to_index()

    patch_cols= {} # channel -> aligned pandas column

    #===== Main Loop Over Channels =====

    for channel, (device, localID) in channel_ref_dict.items():

        reg_addr= channel_type_registerIDs[type_key][localID]                                                   # get reg addr for localID for given type_key

        if device not in list(dict_of_devices_registers_dfs_dicts.keys()):
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no device data; skipping.')
            continue

        if reg_addr in list(dict_of_devices_registers_dfs_dicts[device].keys()):
            source_col= dict_of_devices_registers_dfs_dicts[device][reg_addr][localID]                          # Get source column (pd Series-like) indexed by Time
        else:
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no register address {reg_addr}; skipping.')
            continue

        # source_col= dict_of_devices_registers_dfs_dicts[device][reg_addr][localID]  # Get source column (pd Series-like) indexed by Time

        time_index= ctda_time.intersection(source_col.index)                                                    # Aligns Time by intersection with ctda.Time
        if len(time_index) == 0:                                                                                # Skip if no overlap in Time
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no overlapping Time with channel_type_da; skipping.')
            continue

        vals= source_col.reindex(time_index)                                                                    # Re-index to aligned Time, keeps only rows with timestamps present in time_index and with same order
        if fill_value is not None:
            vals= vals.fillna(fill_value)                                                                       # Optional fill within aligned rows
        
        vals= vals.dropna()                                                                                     # Drop NaNs to avoid overwriting with NaN
        if vals.empty:                                                                                          # Skip if no values remain after dropna
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no values after alignment; skipping.')
            continue

        if channel not in ctda_channels:                                                                        # Skip if channel not in ctda_channels
            if verbose:
                print(f'Channel {channel} not found in channel_type_da; skipping.')
            continue

        patch_cols[channel]= vals.rename(channel)                                                               # Store aligned column for this channel

    if not patch_cols:
        if verbose:
            print("No valid data found to update channel_type_da; returning original.")
        return ctda
    
    #===== Construct Patch DataFrame =====
    patch_df= pd.concat(patch_cols.values(), axis=1)                                                            # Combine all columns into a DataFrame indexed by Time
    patch_df= patch_df.reindex(index= ctda_time, 
                               columns= [c for c in ctda_channels if c in patch_df.columns])                    # Reindex to ctda_time and ctda_channels, keeps only channels present in patch_df
    
    if verbose:
        print(f'Constructed patch DataFrame with shape: {patch_df.shape} for updating channel_type_da.')
    #===== Update DataArray =====

    #===== Update DataArray (only write where values exist) =====
    for ch in patch_df.columns:
        col = patch_df[ch].dropna()
        if not col.empty:
            ctda.loc[dict(Time=col.index, channel=ch)] = col.astype(ctda.dtype).values

    if verbose:
        print(f'Updated channel_type_da with data from {len(patch_cols)} channels.')
        print(f'New shape: {ctda.shape}')

    return ctda


In [40]:
import numpy as np
import pandas as pd
import xarray as xr

updated_da= update_channel_type_da(
    channel_type_da= test_da,
    channel_ref_dict= channel_ref_dict,
    dict_of_devices_registers_dfs_dicts= device_dfs_dict,
    channel_type_registerIDs= channel_type_registerIDs,
    type_key= type_key,
    fill_value= 0.0,
    verbose= True
)

updated_da= updated_da.fillna(0.0)
display(updated_da.to_dataframe().unstack('channel').drop(columns=['device', 'localID']))

Constructed patch DataFrame with shape: (54, 18) for updating channel_type_da.
Updated channel_type_da with data from 18 channels.
New shape: (54, 18)


Activations_data_array                                          \
channel                    Nosepoke0 Nosepoke1 Nosepoke2 Nosepoke3 Nosepoke4   
Time                                                                           
105136.375456                    0.0       0.0       0.0       0.0       0.0   
105136.670336                    0.0       0.0       0.0       0.0       0.0   
105164.753920                    0.0       0.0       0.0       0.0       0.0   
105164.971648                    0.0       0.0       0.0       0.0       0.0   
105451.754080                    0.0       0.0       0.0       0.0       0.0   
105451.893088                    0.0       0.0       0.0       0.0       0.0   
105539.656512                    0.0       0.0       0.0       0.0       0.0   
105539.954304                    0.0       0.0       0.0       0.0       0.0   
105556.648256                    0.0       0.0       0.0       0.0       0.0   
105556.684160                    0.0       0.0       0.0       0.0       0.0   
105556.701728                    0.0       0.0       0.0       0.0       0.0   
105556.788480                    0.0       0.0       0.0       0.0       0.0   
105633.255072                    1.0       0.0       0.0       0.0       0.0   
105633.568576                    0.0       0.0       0.0       0.0       0.0   
105673.389504                    1.0       0.0       0.0       0.0       0.0   
105673.471680                    0.0       0.0       0.0       0.0       0.0   
105697.369472                    0.0       0.0       0.0       0.0       1.0   
105697.573664                    0.0       0.0       0.0       0.0       0.0   
105766.290784                    0.0       0.0       0.0       0.0       0.0   
105766.619072                    0.0       0.0       0.0       0.0       0.0   
105782.499168                    0.0       0.0       0.0       0.0       0.0   
105782.553984                    0.0       0.0       0.0       0.0       0.0   
105786.995264                    0.0       0.0       0.0       0.0       0.0   
105787.055936                    0.0       0.0       0.0       0.0       0.0   
105790.470816                    0.0       0.0       0.0       0.0       0.0   
105790.591968                    0.0       0.0       0.0       0.0       0.0   
105790.608864                    0.0       0.0       0.0       0.0       0.0   
105790.743616                    0.0       0.0       0.0       0.0       0.0   
105792.610464                    0.0       0.0       0.0       0.0       0.0   
105792.791680                    0.0       0.0       0.0       0.0       0.0   
105798.357952                    0.0       0.0       0.0       0.0       0.0   
105798.541792                    0.0       0.0       0.0       0.0       0.0   
105799.842304                    0.0       0.0       0.0       0.0       0.0   
105799.906944                    0.0       0.0       0.0       0.0       0.0   
105850.543296                    0.0       0.0       0.0       0.0       0.0   
105850.566944                    0.0       0.0       0.0       0.0       0.0   
105850.584288                    0.0       0.0       0.0       0.0       0.0   
105850.825312                    0.0       0.0       0.0       0.0       0.0   
105860.990208                    0.0       0.0       0.0       0.0       0.0   
105861.171392                    0.0       0.0       0.0       0.0       0.0   
105883.071616                    0.0       0.0       0.0       0.0       0.0   
105883.147104                    0.0       0.0       0.0       0.0       0.0   
105908.860672                    0.0       0.0       0.0       0.0       0.0   
105909.231936                    0.0       0.0       0.0       0.0       0.0   
105924.389280                    0.0       0.0       0.0       0.0       0.0   
105924.718752                    0.0       0.0       0.0       0.0       0.0   
105935.863360                    0.0       0.0       0.0       0.0       0.0   
105935.956160 

In [41]:
## Get every timepoint where a nosepoke is true in device_df_dict

from matplotlib.pylab import f
for device, regs in device_dfs_dict.items():
    df = regs.get('32') if '32' in regs else regs.get(32)
    if df is None:
        print(f"{device}: register 32 not found")
        continue

    # Prefer known activation localIDs if present, else use all columns
    cols = [c for c in df.columns if c in Nosepoke_Activations_localIDs] or list(df.columns)

    print(f"== Device {device} (register 32) ==")
    for col in cols:
        # rows where this column is True
        true_rows = df.loc[df[col].astype(bool), col]
        if true_rows.empty:
            continue

        print(f"  {col}:")
        for ts, val in true_rows.items():
            print(f"    {ts}: {val}")


display(device_dfs_dict['Behavior0']['32'])
display(device_dfs_dict['Behavior1']['32'])
display(device_dfs_dict['Behavior2']['32'])
display(device_dfs_dict['Behavior3']['32'])
display(device_dfs_dict['Behavior4']['32'])
display(device_dfs_dict['Behavior5']['32'])

== Device Behavior0 (register 32) ==
  DIPort0:
    105633.255072: True
    105673.389504: True
  DIPort1:
    106035.302304: True
  DIPort2:
    106043.345664: True
== Device Behavior1 (register 32) ==
  DIPort1:
    105697.369472: True
  DIPort2:
    105556.648256: True
    105556.701728: True
== Device Behavior2 (register 32) ==
  DIPort1:
    105136.375456: True
    105164.75392: True
  DIPort2:
    105790.470816: True
    105790.608864: True
    105792.610464: True
== Device Behavior3 (register 32) ==
  DIPort0:
    105766.290784: True
    105782.499168: True
    105786.995264: True
  DIPort1:
    105798.357952: True
    105799.842304: True
  DIPort2:
    105451.75408: True
    105908.860672: True
== Device Behavior4 (register 32) ==
  DIPort0:
    105850.543296: True
    105850.584288: True
  DIPort1:
    105883.071616: True
  DIPort2:
    105924.38928: True
== Device Behavior5 (register 32) ==
  DIPort0:
    105539.656512: True
    105860.990208: True
    105935.86336: True
  DI

,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105633.255072,True,False,False,False
105633.568576,False,False,False,False
105673.389504,True,False,False,False
105673.471680,False,False,False,False
106035.302304,False,True,False,False
106035.322752,False,False,False,False
106043.345664,False,False,True,False
106043.443744,False,False,False,False


,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105556.648256,False,False,True,False
105556.684160,False,False,False,False
105556.701728,False,False,True,False
105556.788480,False,False,False,False
105697.369472,False,True,False,False
105697.573664,False,False,False,False


,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105136.375456,False,True,False,False
105136.670336,False,False,False,False
105164.753920,False,True,False,False
105164.971648,False,False,False,False
105790.470816,False,False,True,False
105790.591968,False,False,False,False
105790.608864,False,False,True,False
105790.743616,False,False,False,False
105792.610464,False,False,True,False


,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105451.754080,False,False,True,False
105451.893088,False,False,False,False
105766.290784,True,False,False,False
105766.619072,False,False,False,False
105782.499168,True,False,False,False
105782.553984,False,False,False,False
105786.995264,True,False,False,False
105787.055936,False,False,False,False
105798.357952,False,True,False,False


,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105850.543296,True,False,False,False
105850.566944,False,False,False,False
105850.584288,True,False,False,False
105850.825312,False,False,False,False
105883.071616,False,True,False,False
105883.147104,False,False,False,False
105924.389280,False,False,True,False
105924.718752,False,False,False,False


,DIPort0,DIPort1,DIPort2,DI3
Time,,,,
105539.656512,True,False,False,False
105539.954304,False,False,False,False
105860.990208,True,False,False,False
105861.171392,False,False,False,False
105935.863360,True,False,False,False
105935.956160,False,False,False,False
106094.423488,False,True,False,False
106094.554592,False,False,False,False


In [42]:


# # def update_base_da(
# #     data_array: xr.DataArray = None,
# #     virtual_map_dict: dict = None,
# #     device_dfs_dict: dict = None,
# #     virtual_key_map: dict = None,   # {source_key: {localID: reg_address}}
# #     source_key: str = None,
# #     fill_value=None,
# #     verbose: bool = False,
# # ):
# #     """
# #     Update data_array at (Time, unified_dim) positions using:
# #       device_dfs_dict[device][reg_address][localID]
# #     Robust to dim order changes by inferring dim names.
# #     """
# #     # ===== Error Checks =====
# #     if device_dfs_dict is None:
# #         raise ValueError("device_dfs_dict is required for this update path.")
# #     if virtual_key_map is None or source_key is None:
# #         raise ValueError("virtual_key_map and source_key are required.")
# #     if data_array is None:
# #         raise ValueError("data_array is required.")
# #     if virtual_map_dict is None:
# #         raise ValueError("virtual_map_dict is required.")
# #     if not isinstance(data_array, xr.DataArray):
# #         raise ValueError("data_array must be an xarray DataArray.")

# #     da = data_array

# #     # ===== Infer dims safely (no positional indexing) =====
# #     if "Time" not in da.dims:
# #         raise ValueError(f"Expected a 'Time' dim, got da.dims={da.dims}")
# #     time_dim = "Time"

# #     other_dims = [d for d in da.dims if d != time_dim]
# #     if len(other_dims) != 1:
# #         raise ValueError(f"Expected exactly 2 dims (Time + unified). Got da.dims={da.dims}")
# #     unified_dim = other_dims[0]

# #     da_time = da.get_index(time_dim)
# #     da_unified = da.get_index(unified_dim)

# #     if verbose:
# #         print(f"Updating DataArray dims={da.dims} (time_dim={time_dim}, unified_dim={unified_dim})")

# #     patch_cols = {}  # unified_coord -> aligned Series (indexed by Time)

# #     for unified_coord, entry in virtual_map_dict.items():
# #         device, localID = entry
# #         reg_addr = virtual_key_map[source_key][localID]

# #         if device not in device_dfs_dict:
# #             continue
# #         if reg_addr not in device_dfs_dict[device]:
# #             continue
# #         if localID not in device_dfs_dict[device][reg_addr].columns:
# #             continue

# #         source_col = device_dfs_dict[device][reg_addr][localID]  # Series indexed by Time
# #         time_index = da_time.intersection(source_col.index)
# #         if len(time_index) == 0:
# #             continue

# #         vals = source_col.reindex(time_index)
# #         if fill_value is not None:
# #             vals = vals.fillna(fill_value)
# #         vals = vals.dropna()
# #         if vals.empty:
# #             continue

# #         if unified_coord not in da_unified:
# #             continue

# #         patch_cols[unified_coord] = vals.rename(unified_coord)

# #     if not patch_cols:
# #         if verbose:
# #             print("No columns to update; returning original.")
# #         return da

# #     patch_df = pd.DataFrame(patch_cols)  # index=Time, columns=unified

# #     # ===== Correct assignment (explicit dim names) =====
# #     da.loc[{time_dim: patch_df.index, unified_dim: patch_df.columns}] = patch_df.to_numpy()
# #     return da



# updated_da= update_base_da(data_array= test_da,
#                             virtual_map_dict= channel_ref_dict,
#                             device_dfs_dict= device_dfs_dict,
#                             virtual_key_map= channel_type_registerIDs,
#                             source_key= type_key,
#                             fill_value= None,
#                             verbose= False

# )
# display(updated_da.to_dataframe().unstack('channel').drop(columns=['device', 'localID']))

In [43]:
def construct_base_da(values: list | np.ndarray | None= None,
                      unified_dim: list = None,
                      unified_coord_name: str= 'unified_coord',
                      virtual_coord_names: list= None,
                      virtual_map_dict: dict= None,
                      dict_of: str= 'dicts',
                      times: list | np.ndarray= None,
                      name: str= 'base_dataarray',
                      test_help_values: bool= False,
                      verbose: bool= False
                      ) -> xr.DataArray:
    '''
    Construct a base/placeholder DataArray for a set of given unified/global coordinates for all timepoints and their associated virtual coordinates.
    -----------------------------------

    Parameters:
        values (list or np.ndarray): Values to populate the DataArray. If None, a placeholder array of NaNs will be created.
        unified_dim (list): List of unified/global coordinate names to include in the DataArray.
        virtual_coord_names (list): List of virtual coordinate dimensions to include in the DataArray.
        virtual_map_dict (dict): Dictionary mapping unified/global coordinates to their virtual coordinate information.
        dict_of (str): Specifies the format of virtual_map_dict. Options are 'dicts' or 'tuples'.
        times (list): List of timepoints to include in the DataArray. This should be obtained using collect_timestamps_dict with return_type='list'.
        name (str): Name of the DataArray.
        test_help_values (bool): If True, use test helper values instead of the main values.
        verbose (bool): If True, print verbose output.
    
    Returns:
        A DataArray populated with the specified values, or a placeholder array if no values are provided.
    '''
    
    from collections.abc import Mapping

    dict_of= dict_of.lower()

    if dict_of not in ("dicts", "tuples"):
        raise ValueError(f"dict_of must be either 'dicts' or 'tuples', got: {dict_of}")

    if times is None:
        raise ValueError("times must be provided.")
    if unified_dim is None:
        raise ValueError("unified_dim must be provided.")
    if virtual_coord_names is None:
        raise ValueError("virtual_coord_names must be provided.")
    if virtual_map_dict is None:
        raise ValueError("virtual_map_dict must be provided.")
    
    missing= [u for u in unified_dim if u not in virtual_map_dict]
    if missing: raise KeyError(f'Error: virtual_map_dict is missing entries for unified coordinates: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    
    if dict_of == "tuples" and any(isinstance(v, dict) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='tuples' but at least one entry is a dict. Use dict_of='dicts'.")

    if dict_of == "dicts" and any(isinstance(v, (tuple, list)) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='dicts' but at least one entry is a tuple/list. Use dict_of='tuples'.")



    #=== Create data to populate DataArray 
    if values is None and test_help_values is False:
        if verbose: print(f'No values provided, test_help_values is False. Creating NaNs placeholder DataArray with {len(times)} timestamps and {len(unified_dim)} unified coordinates.')
        values= np.zeros((len(times), len(unified_dim)), dtype= bool) 

    #=== Create test helper values if specified
    if values is None and test_help_values is True:
        if verbose: print(f'No values provided, but test_help_values is True. Creating test helper DataArray with {len(times)} timestamps and {len(unified_dim)} unified coordinates.')
        value_list = []
        for _ in times:
            time_values = []
            for unified in unified_dim:
                virtual_coords = virtual_map_dict[unified]

                if dict_of == 'dicts':
                    if not isinstance(virtual_coords, Mapping): raise TypeError(f'Expected virtual_map_dict to be dict-of-dicts when dict_of="dicts", but got {type(virtual_coords)} for unified coordinate {unified}')
                    
                    coord_str = ', '.join([f'{k}: {v}' for k, v in virtual_coords.items()])

                elif dict_of == 'tuples':
                    
                    if not isinstance(virtual_coords, (tuple, list)): raise TypeError(f'Expected virtual_map_dict to be dict-of-tuples/lists when dict_of="tuples", but got {type(virtual_coords)} for unified coordinate {unified}')
                    
                    if len(virtual_coords) != len(virtual_coord_names): raise ValueError(f"Tuple length mismatch for {unified}: len(entry)={len(virtual_coords)} vs len(virtual_coord_names)={len(virtual_coord_names)}")
                    
                    coord_str = ", ".join([f"{k}: {v}" for k, v in zip(virtual_coord_names, virtual_coords)])

                else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

                time_values.append(f'({unified}, {coord_str})')

            value_list.append(time_values)
        values = np.array(value_list)
        if verbose: print(f'Constructed test helper values array with shape: {values.shape}')
    
    if values is not None and test_help_values is True:
        print('Warning: Both values provided and test_help_values is True. Using provided values.')
    
    #=== Build coordinates 
    
    # Primary/Base coordinates
    coords= {
        'Time': times,
        unified_coord_name: unified_dim,
        }
    
    # Virtual coordinates (dict-of-dicts / named fields)
    if dict_of == 'dicts':
        for vcoord in virtual_coord_names:
            try: vcoord_values= [virtual_map_dict[unified][vcoord] for unified in unified_dim]
            except KeyError as e:
                raise KeyError(f'Missing key {e} in virtual_map_dict entries (dict_of="dicts")') from e
            
            coords[vcoord] = (unified_coord_name, vcoord_values)
    
    
    # Virtual coordinates (dict-of-tuples / positional fields)
    elif dict_of == 'tuples':
        for i, vcoord in enumerate(virtual_coord_names):
            try: vcoord_values = [virtual_map_dict[unified][i] for unified in unified_dim]
            except IndexError as e:
                raise IndexError(f'Index {i} out of range for virtual_map_dict entries (dict_of="tuples")') from e
            
            coords[vcoord] = (unified_coord_name, vcoord_values)
    
    else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

    ########################################################
    """
    NOTE: virtual_map_dict can be either:
      (A) dict-of-dicts (recommended):
          virtual_map_dict[unified] = {"device": ..., "localID": ..., ...}
          -> use key lookup: virtual_map_dict[unified][vcoord]
    
      (B) dict-of-tuples/lists (positional):
          virtual_map_dict[unified] = (device, localID, ...)
          -> assumes tuple order matches virtual_coord_names
    
    Example for (B):
    for i, vcoord in enumerate(virtual_coord_names):
        vcoord_values = [virtual_map_dict[unified][i] for unified in unified_dim]
        coords[vcoord] = (unified_coord_name, vcoord_values)
    
    Dict-of-tuples requires that the order of elements in the tuple matches the order in virtual_coord_names.
    """
    ########################################################

    #=== Create DataArray
    da= xr.DataArray(
        data=values,
        dims= ['Time', unified_coord_name],
        coords= coords,
        name= name,
        attrs= {
            'description': f'Base DataArray for unified coordinates of type {name}',
            'source': 'Constructed using construct_base_da function'
        }
    )
    return da

In [44]:
# Wrapper for converting construct_base_da to function as construct_channel_type_da

def construct_channel_type_da(values: list | np.ndarray | None = None,
                                test_help_values: bool= False,
                                channels: list= None,
                                channels_ref_dict: dict= None,
                                times: list | np.ndarray= None,
                                name: str= 'channel_type_da',
                                verbose: bool= True,
                                dict_of: str= 'tuples'
                                ) -> xr.DataArray:
    '''
    Construct a Placeholder DataArray for a set of given channels for all timepoints across all devices for a specific register type. Uses construct_base_da internally.
    -----------------------------------
    Parameters:
        values (list or np.ndarray): Values to populate the DataArray. If None, a placeholder array of NaNs will be created.
        test_help_values (bool): If True, use test helper values instead of the main values.
        channels (list): List of channel names to include in the DataArray.
        channels_ref_dict (dict): Dictionary mapping channel names to their reference information.
        times (list): List of timepoints to include in the DataArray. This should be obtained using get_df_dict_timestamps with return_type='list'.
        name (str): Name of the DataArray.
    Returns:
        A DataArray populated with the specified values, or a placeholder array if no values are provided.
    '''
    if verbose: print(f'Warning: construct_channel_type_da is deprecated. Use construct_base_da instead with appropriate parameters.')
    return construct_base_da(
        values= values if values is not None else None,
        unified_dim= channels,
        unified_coord_name= 'channel',
        virtual_coord_names= ['device', 'localID'],
        virtual_map_dict= channels_ref_dict,
        dict_of= dict_of,
        times= times,
        name= name,
        test_help_values= test_help_values,
        verbose= verbose
    )






In [45]:
virtual_map= {'global_1': {'vir_1': 'a', 'vir_2': 0, 'vir_3': 'aa'},
                'global_2': {'vir_1': 'b', 'vir_2': 1, 'vir_3': 'bb'},
                'global_3': {'vir_1': 'c', 'vir_2': 2, 'vir_3': 'cc'},
                'global_4': {'vir_1': 'd', 'vir_2': 3, 'vir_3': 'dd'},
                'global_5': {'vir_1': 'e', 'vir_2': 4, 'vir_3': 'ee'}

              }

# lookup array should have shape (5, 4) with dims (unified_coord_name, virtual_coord_names)

import numpy as np
import xarray as xr

unifieds = list(virtual_map.keys())
vir_1_vals = sorted({v['vir_1'] for v in virtual_map.values()})
vir_2_vals = sorted({v['vir_2'] for v in virtual_map.values()})
vir_3_vals = sorted({v['vir_3'] for v in virtual_map.values()})

lookup_arr = np.zeros((len(unifieds), len(vir_1_vals), len(vir_2_vals), len(vir_3_vals)), dtype=bool)

for i, u in enumerate(unifieds):
    v1 = vir_1_vals.index(virtual_map[u]['vir_1'])
    v2 = vir_2_vals.index(virtual_map[u]['vir_2'])
    v3 = vir_3_vals.index(virtual_map[u]['vir_3'])
    lookup_arr[i, v1, v2, v3] = True

da = xr.DataArray(
    lookup_arr,
    dims=['unified', 'vir_1', 'vir_2', 'vir_3'],
    coords={'unified': unifieds, 'vir_1': vir_1_vals, 'vir_2': vir_2_vals, 'vir_3': vir_3_vals},
    name='lookup_matrix'
)
df = da.to_dataframe()
df


lookup_matrix
unified  vir_1 vir_2 vir_3               
global_1 a     0     aa              True
                     bb             False
                     cc             False
                     dd             False
                     ee             False
...                                   ...
global_5 e     4     aa             False
                     bb             False
                     cc             False
                     dd             False
                     ee              True

[625 rows x 1 columns]

In [46]:


device_list= ['Behavior0',
               'Behavior1',
               'Behavior2',
               'Behavior3',
               'Behavior4',
               'Behavior5',
]

device_IDs= {
    f'{device_list[0]}': 'ID_0',
    f'{device_list[1]}': 'ID_1',
    f'{device_list[2]}': 'ID_2',
    f'{device_list[3]}': 'ID_3',
    f'{device_list[4]}': 'ID_4',
    f'{device_list[5]}': 'ID_5',
}

device_registers_dict= {
    'Behavior0': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior1': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior2': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior3': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior4': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior5': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
}

Nosepoke_Activations_localIDs= ['DIPort0',
                                'DIPort1',
                                'DIPort2',
]

Nosepoke_LED_Activations_localIDs= ['DOPort0',
                                    'DOPort1',
                                    'DOPort2',
]

Nosepoke_Valve_Activations_localIDs= ['SupplyPort0',
                                      'SupplyPort1',
                                      'SupplyPort2',
]

Nosepoke_Reward_Release_localIDs= ['PulseSupplyPort0',
                                   'PulseSupplyPort1',
                                   'PulseSupplyPort2',
]

channel_list= ['Nosepoke0',
               'Nosepoke1',
               'Nosepoke2',
               'Nosepoke3',
               'Nosepoke4',
               'Nosepoke5',
               'Nosepoke6',
               'Nosepoke7',
               'Nosepoke8',
               'Nosepoke9',
               'Nosepoke10',
               'Nosepoke11',
               'Nosepoke12',
               'Nosepoke13',
               'Nosepoke14',
               'Nosepoke15',
               'Nosepoke16',
               'Nosepoke17',]


channel_type_localIDs= {
    'Activations': Nosepoke_Activations_localIDs,
    'LED_Activations': Nosepoke_LED_Activations_localIDs,
    'Valve_Activations': Nosepoke_Valve_Activations_localIDs,
    'Reward_Release': Nosepoke_Reward_Release_localIDs,
}

Activations_localID_register_dict= {
    'DIPort0': '32',
    'DIPort1': '32',
    'DIPort2': '32',
}
LED_Activations_localID_register_dict= {
    'DOPort0': '34',
    'DOPort1': '34',
    'DOPort2': '34',
}

Valve_Activations_localID_register_dict= {
    'SupplyPort0': '34',
    'SupplyPort1': '34',
    'SupplyPort2': '34',
}

Reward_Release_localID_register_dict= {
    'PulseSupplyPort0': '49',
    'PulseSupplyPort1': '50',
    'PulseSupplyPort2': '51',
}

channel_type_registerIDs= {
    'Activations': Activations_localID_register_dict,
    'LED_Activations': LED_Activations_localID_register_dict,
    'Valve_Activations': Valve_Activations_localID_register_dict,
    'Reward_Release': Reward_Release_localID_register_dict,
}

type_key= list(channel_type_localIDs.keys())[0]  # Type of channel to process (e.g., 'Activations'). Must be exact prefix from channel_type_localIDs keys

channel_ref_dict= {
    f'{channel_list[0]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[1]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[2]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[3]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[4]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[5]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[6]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[7]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[8]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[9]}':  (f'{device_list[3]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[10]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[11]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[12]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[13]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[14]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[15]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[16]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[17]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][2]}'),
}

# channel_ref_dict= {
#                 f'{channel_list[0]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[1]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[2]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[3]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[4]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[5]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[6]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[7]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[8]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[9]}':  {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[10]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[11]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[12]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[13]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[14]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[15]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[16]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[17]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},
#             }
# channel_dict= {channel: (device, localID) for channel, (device, localID) in channel_ref_dict.items()}


device_type= 'Behavior'  # Type of device to process (e.g., 'Behavior'). Must be exact prefix 

channel_type_da_name= f'{type_key}'
print(f'Processing channel type: {channel_type_da_name}')

Processing channel type: Activations


In [47]:
from zmq import device
from Refactor.Timestamps.timestamps import collect_timestamps_dict, collect_timestamps_nested_dict


# times= collect_timestamps_nested_dict(dfs_dict= device_dfs_dict, return_type= 'list')
times= collect_timestamps_nested_dict(dfs_dict= device_dfs_dict,
                                      data_key= type_key,
                                      peripheral_registerIDs= channel_type_registerIDs,
                                      return_type= 'list')
print(len(times))
# test construct_channel_type_da
test_da= construct_channel_type_da(
    values= None,
    test_help_values= True,
    channels= channel_list,
    channels_ref_dict= channel_ref_dict,
    times= times,
    name= channel_type_da_name,
    verbose= True,
    dict_of= 'tuples'
)
print(test_da)

display(test_da.to_dataframe().unstack('channel').drop(columns=['device', 'localID']))

54
No values provided, but test_help_values is True. Creating test helper DataArray with 54 timestamps and 18 unified coordinates.
Constructed test helper values array with shape: (54, 18)
<xarray.DataArray 'Activations' (Time: 54, channel: 18)> Size: 191kB
array([['(Nosepoke0, device: Behavior0, localID: DIPort0)',
        '(Nosepoke1, device: Behavior0, localID: DIPort1)',
        '(Nosepoke2, device: Behavior0, localID: DIPort2)',
        '(Nosepoke3, device: Behavior1, localID: DIPort0)',
        '(Nosepoke4, device: Behavior1, localID: DIPort1)',
        '(Nosepoke5, device: Behavior1, localID: DIPort2)',
        '(Nosepoke6, device: Behavior2, localID: DIPort0)',
        '(Nosepoke7, device: Behavior2, localID: DIPort1)',
        '(Nosepoke8, device: Behavior2, localID: DIPort2)',
        '(Nosepoke9, device: Behavior3, localID: DIPort0)',
        '(Nosepoke10, device: Behavior3, localID: DIPort1)',
        '(Nosepoke11, device: Behavior3, localID: DIPort2)',
        '(Nosepoke12

Activations  \
channel                                               Nosepoke0   
Time                                                              
105136.375456  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105136.670336  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105164.753920  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105164.971648  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105451.754080  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105451.893088  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105539.656512  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105539.954304  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.648256  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.684160  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.701728  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105556.788480  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105633.255072  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105633.568576  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105673.389504  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105673.471680  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105697.369472  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105697.573664  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105766.290784  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105766.619072  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105782.499168  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105782.553984  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105786.995264  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105787.055936  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.470816  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.591968  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.608864  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105790.743616  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105792.610464  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105792.791680  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105798.357952  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105798.541792  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105799.842304  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105799.906944  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.543296  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.566944  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.584288  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105850.825312  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105860.990208  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105861.171392  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105883.071616  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105883.147104  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105908.860672  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105909.231936  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105924.389280  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105924.718752  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105935.863360  (Nosepoke0, device: Behavior0, localID: DIPort0)   
105935.956160  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106035.302304  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106035.322752  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106043.345664  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106043.443744  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106094.423488  (Nosepoke0, device: Behavior0, localID: DIPort0)   
106094.554592  (Nosepoke0, device: Behavior0, localID: DIPort0)   

                                                                 \
channel                                               Nosepoke1   
Time                                                              
105136.375456  (Nosepoke1, devi

In [48]:
def construct_lookup_array(unified_dim: list= None,
                           unified_coord_name: str= 'unified_coord',
                           virtual_coord_names: list= None,
                           virtual_map_dict: dict= None,
                           dict_of: str= 'dicts',
                           name: str= 'lookup_matrix',
                           verbose: bool= False
                           ) -> tuple[xr.DataArray, dict[str, list]]:
    '''
    Construct a Lookup Array DataArray for a set of given unified/global coordinates and their associated virtual coordinates.
    -----------------------------------

    Parameters:
        unified_dim (list): List of unified/global coordinate names to include in the DataArray.
        unified_coord_name (str): Name for the unified/global coordinate dimension.
        virtual_coord_names (list): List of virtual coordinate dimensions to include in the DataArray.
        virtual_map_dict (dict): Dictionary mapping unified/global coordinates to their virtual coordinate information.
        dict_of (str): Specifies the format of virtual_map_dict. Options are 'dicts' or 'tuples'.
        name (str): Name of the DataArray.
        verbose (bool): If True, prints additional information. 
    
    Returns:
        lookup_da (xr.DataArray): A DataArray representing the lookup matrix.
        lookup_virtual_coords (dict): A dictionary containing lists of unique virtual coordinate values for each virtual coordinate dimension. Keys are virtual coordinate names, values are lists of unique values. 
    '''
    #==== Input validation and preprocessing ====
    dict_of= dict_of.lower()

    if dict_of not in ("dicts", "tuples"):
        raise ValueError(f"dict_of must be either 'dicts' or 'tuples', got: {dict_of}")
    
    if unified_dim is None or virtual_coord_names is None or virtual_map_dict is None:
        raise ValueError("unified_dim, virtual_coord_names, and virtual_map_dict must be provided.")
    
    if len(unified_dim) == 0:
        raise ValueError("unified_dim cannot be empty.")
    if len(virtual_coord_names) == 0:
        raise ValueError("virtual_coord_names cannot be empty.")
    
    if verbose: print(f'Constructing lookup array for {len(unified_dim)} unified coordinates and virtual coordinates: {virtual_coord_names}')
    
    missing= [u for u in unified_dim if u not in virtual_map_dict]
    if missing: raise KeyError(f'Error: virtual_map_dict is missing entries for unified coordinates: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    
    if dict_of == "tuples" and any(isinstance(v, dict) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='tuples' but at least one entry is a dict. Use dict_of='dicts'.")

    if dict_of == "dicts" and any(isinstance(v, (tuple, list)) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='dicts' but at least one entry is a tuple/list. Use dict_of='tuples'.")

    #==== Collect Unique Values for Each Virtual Coordinate (Levels)
    lookup_virtual_coords= {vcoord: [] for vcoord in virtual_coord_names}

    for unified in unified_dim:
        virtual_coords= virtual_map_dict[unified]

        if dict_of == 'tuples':
            if not isinstance(virtual_coords, (tuple, list)):
                raise TypeError(f'Expected virtual_map_dict to be dict-of-tuples/lists when dict_of="tuples", but got {type(virtual_coords)} for unified coordinate {unified}')
            if len(virtual_coords) != len(virtual_coord_names):
                raise ValueError(f"Tuple length mismatch for {unified}: len(entry)={len(virtual_coords)} vs len(virtual_coord_names)={len(virtual_coord_names)}")
            
            for vcoord_name, vcoord_value in zip(virtual_coord_names, virtual_coords):
                if vcoord_value not in lookup_virtual_coords[vcoord_name]:
                    lookup_virtual_coords[vcoord_name].append(vcoord_value)

        elif dict_of == 'dicts':
            if not isinstance(virtual_coords, Mapping):
                raise TypeError(f'Expected virtual_map_dict to be dict-of-dicts when dict_of="dicts", but got {type(virtual_coords)} for unified coordinate {unified}')
            
            for vcoord_name in virtual_coord_names:
                if vcoord_name not in virtual_coords:
                    raise KeyError(f'Missing key {vcoord_name} in virtual_map_dict entry for unified coordinate {unified} (dict_of="dicts")')
                
                vcoord_value= virtual_coords[vcoord_name]
                if vcoord_value not in lookup_virtual_coords[vcoord_name]:
                    lookup_virtual_coords[vcoord_name].append(vcoord_value)
        
        else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

        if verbose: 
            for vcoord_name in virtual_coord_names: 
                print(f'Collected {vcoord_name}: {lookup_virtual_coords[vcoord_name]} unique values.')
        

    #==== Construct Lookup Array ====
    
    # Fast lookup tables from virtual coordinate values to integer indices along each virtual dimension
    index_map= {
        vcoord: {val: idx for idx, val in enumerate(lookup_virtual_coords[vcoord])} 
        for vcoord in virtual_coord_names
        }

    # Allocate boolean array for lookup matrix
    lookup_shape= [len(unified_dim)] + [len(lookup_virtual_coords[vcoord]) for vcoord in virtual_coord_names]
    lookup_arr= np.zeros(lookup_shape, dtype= bool)

    # Populate lookup array
    for i, unified in enumerate(unified_dim):
        
        entry= virtual_map_dict[unified]

        if dict_of == 'tuples':
            idx= [i] + [index_map[vcoord_name][vcoord_value] 
            for vcoord_name, vcoord_value in zip(virtual_coord_names, entry)
            ]
        
        elif dict_of == 'dicts':
            idx= [i] + [index_map[vcoord_name][entry[vcoord_name]]
            for vcoord_name in virtual_coord_names
            ]
        else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")
        lookup_arr[tuple(idx)] = True
    
    #==== Create DataArray 
    dims= [unified_coord_name] + virtual_coord_names
    
    coords= {unified_coord_name: unified_dim,}

    coords.update({vcoord: lookup_virtual_coords[vcoord] for vcoord in virtual_coord_names})

    lookup_da= xr.DataArray(
        data= lookup_arr,
        dims= dims,
        coords= coords,
        name= name,
        attrs= {
            'description': 'Lookup matrix indicating presence of unified coordinates across virtual coordinate dimensions',
            'source': 'Constructed using construct_lookup_array function'
        }
    )
    return lookup_da, lookup_virtual_coords

In [49]:
def construct_channel_lookup_da(channels= None,
                                channels_ref_dict= None,
                                dict_of= 'tuples',
                                verbose= False
):
    '''
    Construct a Channel Lookup DataArray for a set of given channels.
    -----------------------------------

    Parameters:
        channels (list): List of channel names to include in the DataArray.
        channels_ref_dict (dict): Dictionary mapping channel names to their reference information.
        dict_of (str): Specifies the format of channels_ref_dict. Options are 'dicts' or 'tuples'.
        verbose (bool): If True, prints additional information.

    Returns:
        lookup_da (xr.DataArray): A DataArray representing the channel lookup matrix.
        lookup_devices (list): List of unique devices found in the channels_ref_dict.
        lookup_localIDs (list): List of unique local IDs found in the channels_ref_dict.
    '''

    if verbose: print(f'Warning: construct_channel_lookup_da is deprecated. Use construct_lookup_array instead with appropriate parameters.')
    lookup_da, lookup_virtual_coords= construct_lookup_array(
        unified_dim= channels,
        unified_coord_name= 'channel',
        virtual_coord_names= ['device', 'localID'],
        virtual_map_dict= channels_ref_dict,
        dict_of= dict_of,
        name= 'channel_lookup_matrix',
        verbose= verbose
    )
    return lookup_da, lookup_virtual_coords['device'], lookup_virtual_coords['localID']

In [50]:
channel_ref_dict= {
    f'{channel_list[0]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[1]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[2]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[3]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[4]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[5]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[6]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[7]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[8]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[9]}':  (f'{device_list[3]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[10]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[11]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[12]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[13]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[14]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][2]}'),

    f'{channel_list[15]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][0]}'),
    f'{channel_list[16]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][1]}'),
    f'{channel_list[17]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][2]}'),
}

# channel_ref_dict= {
#                 f'{channel_list[0]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[1]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[2]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[3]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[4]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[5]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[6]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[7]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[8]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[9]}':  {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[10]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[11]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[12]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[13]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[14]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

#                 f'{channel_list[15]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
#                 f'{channel_list[16]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
#                 f'{channel_list[17]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},
#             }

In [51]:
# Example usage of construct_channel_lookup_da
channel_lookup_da, lookup_devices, lookup_localIDs= construct_channel_lookup_da(
    channels= channel_list,
    channels_ref_dict= channel_ref_dict,
    dict_of= 'tuples',
    verbose= False
)
print(channel_lookup_da)
display(channel_lookup_da.to_dataframe())
print(f'Lookup Devices: {lookup_devices}')
print(f'Lookup Local IDs: {lookup_localIDs}')


<xarray.DataArray 'channel_lookup_matrix' (channel: 18, device: 6, localID: 3)> Size: 324B
array([[[ True, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False]],

       [[False,  True, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False]],

       [[False, False,  True],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False]],
...
       [[False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [ True, False, False]],

       [[False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],

channel_lookup_matrix
channel    device    localID                       
Nosepoke0  Behavior0 DIPort0                   True
                     DIPort1                  False
                     DIPort2                  False
           Behavior1 DIPort0                  False
                     DIPort1                  False
...                                             ...
Nosepoke17 Behavior4 DIPort1                  False
                     DIPort2                  False
           Behavior5 DIPort0                  False
                     DIPort1                  False
                     DIPort2                   True

[324 rows x 1 columns]

Lookup Devices: ['Behavior0', 'Behavior1', 'Behavior2', 'Behavior3', 'Behavior4', 'Behavior5']
Lookup Local IDs: ['DIPort0', 'DIPort1', 'DIPort2']


Select functions

In [52]:
def ulookup(lookup_arr: xr.DataArray = None,
            unified_coord_name: str= 'unified_coord',
            verbose: bool= False,
            **selectors,
        ) -> list:
    '''
    uLookup: Selects unified/global coordinates from a lookup DataArray based on specified criteria across virtual coordinate dimensions. Supports both positional- and label-based indexing and combinations thereof.
    -----------------------------------

    Parameters:
        lookup_arr (xr.DataArray): N-dim Boolean lookup tensor with dimensions like [unified_coord_name] + virtual coordinate dimensions.
        unified_coord_name (str): Name of the unified/global coordinate dimension.
        verbose (bool): If True, prints additional information.
        **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by. 
    
    Returns:
        list: A list of unified/global coordinate names that match the specified criteria.
    '''
    if lookup_arr is None:
        raise ValueError("lookup_arr must be provided.")
    if unified_coord_name not in lookup_arr.dims:
        raise ValueError(f"{unified_coord_name} is not a dimension in the provided lookup_arr.")
    if verbose: print(f'Starting uLookup with unified_coord_name="{unified_coord_name}" and selectors: {selectors}')


    #=== Gather selection criteria
    sel= {}

    sel= {key: value for key, value in selectors.items() if value is not None}
    if verbose: print(f'Selecting unified coordinates with criteria: {sel}')

    #=== Get subset based on selection criteria
    subset= lookup_arr.sel(**sel) if sel else lookup_arr
    if verbose: print(f'Subset shape after selection: {subset.shape}')

    #=== Reduce across non-unified_coord_name dimensions
    other_dims= [d for d in subset.dims if d != unified_coord_name]
    
    if other_dims:
        per_unified_true= subset.any(dim= other_dims)
    else:
        per_unified_true= subset.astype(bool)
    
    if verbose: print(f'Per-unified_coord_name true shape after reduction: {per_unified_true.shape}')
    
    # return per_unified_true[unified_coord_name].values[per_unified_true.values].tolist()

    labels = per_unified_true[unified_coord_name].values
    mask = np.asarray(per_unified_true.values, dtype=bool)
    return labels[mask].tolist()

In [53]:
def select_channels_from_C_D_L(lookup_da: xr.DataArray= None,
                               unified_coord_name: str= 'channel',
                               verbose: bool= True,
                               channel: str= None,
                               device: str= None,
                               localID: str= None) -> list:
    '''
    Select channels from a Channel-Device-LocalID lookup DataArray based on specified criteria. Deprecated: use ulookup instead.
    -----------------------------------
    Parameters:
        lookup_da (xr.DataArray): The Channel-Device-LocalID lookup DataArray.
        unified_coord_name (str): Name of the unified/global coordinate dimension (default is 'channel').
        verbose (bool): If True, prints additional information.
        channel (str): The channel name to filter by (optional).
        device (str): The device name to filter by (optional).
        localID (str): The local ID to filter by (optional).
    '''
    if verbose: print(f'Warning: select_channels_from_C_D_L is deprecated. Use ulookup instead with appropriate parameters.')
    return ulookup(
        lookup_arr= lookup_da,
        unified_coord_name= unified_coord_name,
        verbose= verbose,
        channel= channel,
        device= device,
        localID= localID
    )

In [54]:



# def select_channels_from_C_D_L(lookup_da= None, channel= None, device= None, localID= None):
#     '''
#     Select channels from a Channel-Device-LocalID lookup DataArray based on specified criteria.
#     -----------------------------------

#     Parameters:
#         lookup_da (xr.DataArray): The Channel-Device-LocalID lookup DataArray.
#         channel (str): The channel name to filter by (optional).
#         device (str): The device name to filter by (optional).
#         localID (str): The local ID to filter by (optional).

#     Returns:
#         list: A list of channel names that match the specified criteria.
#     '''

#     sel= {}

#     if channel is not None: sel['channel']= channel
#     if device is not None: sel['device']= device
#     if localID is not None: sel['localID']= localID

#     # Subset
#     subset= lookup_da.sel(**sel) if sel else lookup_da
#     print(f'Subset shape after selection: {subset.shape}')

#     # Reduce across non-channel dimensions
#     other_dims= [d for d in subset.dims if d != 'channel']
#     if other_dims:
#         per_channel_true= subset.any(dim= other_dims)
#     else:
#         per_channel_true= subset.astype(bool)

#     print(f'Per-channel true shape after reduction: {per_channel_true.shape}')

#     return per_channel_true['channel'].values[per_channel_true.values].tolist()


def test_da_indexing(channel_type_da=None, channel_lookup_da=None, channel_list=None, 
                    channel_ref_dict=None, verbose=False, only_errors=False):
    """
    Test slicing and indexing operations for channel data arrays.
    
    Parameters:
    -----------
    channel_type_da : xarray.DataArray
        Data array containing channel type information
        Data array containing channel type information
    channel_lookup_da : xarray.DataArray
        Lookup data array for channels
    channel_list : list
        List of channel names
    channel_ref_dict : dict
        Dictionary with channel reference information
    verbose : bool, default=True
        If True, prints detailed output for each test
    only_errors : bool, default=False
        If True, only reports when errors occur
    
    Returns:
    --------
        dict : Dictionary with test results
    """
    # Initialize the lookup array if needed
    if channel_lookup_da is None:
        lookup_da, devices, local_ids = construct_channel_lookup_da(channels=channel_list, 
                                                                   channels_ref_dict=channel_ref_dict)
    else:
        lookup_da = channel_lookup_da
        # Extract devices and local_ids from lookup_da
        channels = lookup_da.coords['channel'].values.tolist() if channel_list is None else channel_list
        devices = lookup_da.coords['device'].values.tolist() if 'device' in lookup_da.coords else []
        local_ids = lookup_da.coords['localID'].values.tolist() if 'localID' in lookup_da.coords else []
    
    localIDs = local_ids
    test_results = {}
    
    # Helper function to run test case and handle output
    def run_test_case(test_id, description, selection_params):
        try:
            selected_items = select_channels_from_C_D_L(lookup_da, **selection_params)
            test_results[test_id] = {"success": True, "selected_items": selected_items}
            
            if verbose and not only_errors:
                print(f"\n{test_id}) {description}:")
                print(selected_items)
                if channel_type_da is not None:
                    display(channel_type_da.sel(channel=selected_items).to_dataframe())
                    print("\nWithout device and localID columns:")
                    display(channel_type_da.sel(channel=selected_items).to_dataframe().unstack('channel')
                           .drop(columns=['device', 'localID']))
            return True
            
        except Exception as e:
            test_results[test_id] = {"success": False, "error": str(e)}
            if verbose or only_errors:
                print(f"\n{test_id}) {description} - ERROR: {str(e)}")
            return False
    
    # Define all test cases
    test_cases = [
        ("A", "Slice devices by label range, and take the 2nd localID ('DOPort1')", 
         {"device": slice('Behavior1','Behavior5'), "localID": 'DOPort1'}),
        ("B", "Slice localIDs by label range; any device", 
         {"localID": slice('DOPort0','DOPort2')}),
        ("C", "List of devices; list of localIDs", 
         {"device": ['Behavior0','Behavior2'], "localID": ['DOPort0','DOPort1']}),
        ("D", "Slice channels by label range", 
         {"channel": slice('Nosepoke2','Nosepoke6')}),
        ("E", "Exact single cell", 
         {"channel": 'Nosepoke4', "device": 'Behavior1', "localID": 'DOPort1'}),
        ("F", "All channels (wildcard)", 
         {"channel": None, "device": None, "localID": None}),
        ("G", "Slice using numeric index for channels", 
         {"channel": channels[0:2] if len(channels) >= 2 else []}),
        ("H", "Slice using numeric index for devices", 
         {"device": devices[0:2] if len(devices) >= 2 else []}),
        ("I", "Slice using numeric index for localIDs", 
         {"localID": localIDs[0:2] if len(localIDs) >= 2 else []}),
        ("J", "Slice using label range for devices and localIDs", 
         {"device": slice('Behavior1', 'Behavior2'), "localID": slice('DOPort0', 'DOPort2')}),
        ("K", "Slice using numeric index for devices and localIDs", 
         {"device": devices[1:3] if len(devices) >= 3 else [], 
          "localID": localIDs[0:3] if len(localIDs) >= 3 else []}),
        ("L", "Slice using label range for channels", 
         {"channel": slice('Nosepoke2', 'Nosepoke6'), "device": None, "localID": None}),
        ("M", "Slice using numeric index for channels", 
         {"channel": channels[2:7] if len(channels) >= 7 else [], 
          "device": devices[:], "localID": localIDs[:]})
    ]

  

    
    # Run all test cases
    all_success = True
    for test_id, description, params in test_cases:
        success = run_test_case(test_id, description, params)
        all_success = all_success and success
    
    # Report summary
    if verbose:
        success_count = sum(1 for result in test_results.values() if result["success"])
        total_count = len(test_results)
        
        if only_errors:
            if all_success:
                print("\nAll tests passed successfully!")
            else:
                failures = [(test_id, result) for test_id, result in test_results.items() if not result["success"]]
                print(f"\n{len(failures)}/{total_count} tests failed.")
        else:
            print(f"\nTest results: {success_count}/{total_count} tests passed.")
    
    return test_results



# Example usage of test_da_indexing
test_results= test_da_indexing(
    channel_type_da= test_da,
    channel_lookup_da= channel_lookup_da,
    channel_list= channel_list,
    channel_ref_dict= channel_ref_dict,
    verbose= True,
    only_errors= True
)

Starting uLookup with unified_coord_name="channel" and selectors: {'channel': None, 'device': slice('Behavior1', 'Behavior5', None), 'localID': 'DOPort1'}
Selecting unified coordinates with criteria: {'device': slice('Behavior1', 'Behavior5', None), 'localID': 'DOPort1'}

A) Slice devices by label range, and take the 2nd localID ('DOPort1') - ERROR: "not all values found in index 'localID'. Try setting the `method` keyword argument (example: method='nearest')."
Starting uLookup with unified_coord_name="channel" and selectors: {'channel': None, 'device': None, 'localID': slice('DOPort0', 'DOPort2', None)}
Selecting unified coordinates with criteria: {'localID': slice('DOPort0', 'DOPort2', None)}
Subset shape after selection: (18, 6, 0)
Per-unified_coord_name true shape after reduction: (18,)
Starting uLookup with unified_coord_name="channel" and selectors: {'channel': None, 'device': ['Behavior0', 'Behavior2'], 'localID': ['DOPort0', 'DOPort1']}
Selecting unified coordinates with criter

In [55]:

# Example usage of test_da_indexing
test_results= test_da_indexing(
    channel_type_da= test_da,
    channel_lookup_da= channel_lookup_da,
    channel_list= channel_list,
    channel_ref_dict= channel_ref_dict,
    verbose= True,
    only_errors= True
)

Starting uLookup with unified_coord_name="channel" and selectors: {'channel': None, 'device': slice('Behavior1', 'Behavior5', None), 'localID': 'DOPort1'}
Selecting unified coordinates with criteria: {'device': slice('Behavior1', 'Behavior5', None), 'localID': 'DOPort1'}

A) Slice devices by label range, and take the 2nd localID ('DOPort1') - ERROR: "not all values found in index 'localID'. Try setting the `method` keyword argument (example: method='nearest')."
Starting uLookup with unified_coord_name="channel" and selectors: {'channel': None, 'device': None, 'localID': slice('DOPort0', 'DOPort2', None)}
Selecting unified coordinates with criteria: {'localID': slice('DOPort0', 'DOPort2', None)}
Subset shape after selection: (18, 6, 0)
Per-unified_coord_name true shape after reduction: (18,)
Starting uLookup with unified_coord_name="channel" and selectors: {'channel': None, 'device': ['Behavior0', 'Behavior2'], 'localID': ['DOPort0', 'DOPort1']}
Selecting unified coordinates with criter

In [56]:
@xr.register_dataarray_accessor('ul')
class LookupAccessorConstructor:
    '''
    Base class for making custom xarray DataArray accessors for lookup operations with auto-construction of lookup arrays.
    '''

    def __init__(self,
                 data_array: xr.DataArray
                 ):
        
        self._da= data_array

        lookup_array= self.lookup_constructor(lookup_array= None,
                                             unified_coord_name= None,
                                             dict_of= None,
                                             verbose= False,
                                             )
        self.lookup_array= lookup_array



    def lookup_constructor(self,
                           lookup_array: xr.DataArray | None= None,
                           unified_coord_name: str | None = None,
                           dict_of: str | None= None,
                           verbose: bool= False,
                           ):
        
        if lookup_array: 
            self.lookup_array= lookup_array
            if verbose: print(f'Using provided lookup_array with shape {self.lookup_array.shape}.')

        if dict_of is None:
            if verbose: print('No preferred format for "dict_of" provided, defaulting to "tuples".')
            dict_of= 'tuples'
        elif dict_of.lower() not in ('tuples', 'dicts'):
            raise ValueError(f'Invalid dict_of value: {dict_of}. Must be either "tuples" or "dicts".')
        
        if not hasattr(self, 'lookup_array'):
            
            if verbose: print(f'Lookup array not provided, attempting to auto-construct using data array')

            #==== 1) Unified Coordinate Name
            """
            Name of unified/global coordinate dimension in the lookup array. Assumed to be second dimension in the data array, or first dimension not including 'Time' dimension.
            """
            if unified_coord_name is None:
                unified_coord_name= [dim for dim in self._da.dims if dim.lower() != 'time'][0]
                if verbose: print(f'No unified_coord_name provided, inferred as "{unified_coord_name}" from data array dimensions.')
            
            #==== 2) Unified Dimension Values
            """
            List of unified/global coordinate names to include in the DataArray.
            """
            unified_dim = self._da.coords[unified_coord_name].values.tolist()
            if verbose: print(f'Extracted {len(unified_dim)} unified coordinates from data array dimension "{unified_coord_name}".')

            #==== 3) Virtual Coordinate Names
            """
            Names of coordinates that map each unified/global coordinate to its virtual coordinate information.
            Assumed to be all coordinates in the data array except for the unified coordinate and 'Time' dimension.
            """
            virtual_coord_names= [coord for coord in self._da.coords.keys() if coord != unified_coord_name and coord.lower() != 'time']
            if verbose: print(f'Inferred virtual coordinate names: {virtual_coord_names}.')

            #==== 4) Virtual Map Dictionary
            """
            Dictionary mapping unified/global coordinates to their virtual coordinate information. Can be in dict-of-dicts or dict-of-tuples format.
            """
            if dict_of == 'tuples':
                # Fix: Use dict() to create proper dictionary from zip
                virtual_map_dict = dict(zip(
                    unified_dim, 
                    zip(*(self._da.coords[coord].values.tolist() for coord in virtual_coord_names))
                ))
            elif dict_of == 'dicts':
                virtual_map_dict = {
                    unified: {vcoord: self._da.sel({unified_coord_name: unified})[vcoord].item() for vcoord in virtual_coord_names}
                    for unified in unified_dim
                }
            if verbose: print(f'Constructed virtual_map_dict with {len(virtual_map_dict)} entries in format "{dict_of}".')

            #==== 5) Construct Lookup Array
            self.lookup_array, _ = construct_lookup_array(
                unified_dim= unified_dim,
                unified_coord_name= unified_coord_name,
                virtual_coord_names= virtual_coord_names,
                virtual_map_dict= virtual_map_dict,
                dict_of= dict_of,
                name= f'{unified_coord_name}_lookup_matrix',
                verbose= verbose
            )
            if verbose: print(f'Constructed lookup_array with shape {self.lookup_array.shape}.')
        return self.lookup_array
    
    def ul_sel(self,
                verbose: bool= False,
                **selectors,
            ) -> list:
        '''
        uLookup: Selects unified/global coordinates from the lookup DataArray based on specified criteria across virtual coordinate dimensions. Supports both positional- and label-based indexing and combinations thereof.
        -----------------------------------

        Parameters:
            verbose (bool): If True, prints additional information.
            **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by.
        Returns:
            list: A list of unified/global coordinate names that match the specified criteria.
        '''
        u_c_n= [dim for dim in self._da.dims if dim.lower() != 'time'][0]
        return ulookup(
            lookup_arr= self.lookup_array,
            unified_coord_name= u_c_n,
            verbose= verbose,
            **selectors
        )
    
    def sel(self, **selectors) -> list:
        '''
        sel: Alias for ulookup method to select unified/global coordinates based on specified criteria.
        -----------------------------------

        Parameters:
            **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by.
        Returns:
            list: A list of unified/global coordinate names that match the specified criteria.
        '''
        return self.ul_sel(**selectors)
    
    __call__= sel




/tmp/ipykernel_2013889/732022404.py:1: AccessorRegistrationWarning: registration of accessor <class '__main__.LookupAccessorConstructor'> under name 'ul' for type <class 'xarray.core.dataarray.DataArray'> is overriding a preexisting attribute with the same name.
  @xr.register_dataarray_accessor('ul')


In [57]:
# Test lookup_accessor_constructor
selected_udim_accessor = test_da.ul.ul_sel(
    channel='Nosepoke4',
    device='Behavior1',
    localID='DOPort1',
    verbose=False
)
print(f'Selected unified dimensions using accessor: {selected_udim_accessor}')

KeyError: "not all values found in index 'localID'. Try setting the `method` keyword argument (example: method='nearest')."

In [ ]:
# A) Slice devices by label range, and take the 2nd local_id ('m1')
print("A) Slice devices by label range, and take the 2nd local_id ('m1'):")
print(test_da.ul.ul_sel(device=slice('Behavior1','Behavior2'), localID='DOPort1'))

# B) Slice local_ids by label range; any device
print("\nB) Slice local_ids by label range; any device:")
print(test_da.ul.ul_sel(localID=slice('DOPort0','DOPort2')))
# C) List of devices; list of local_ids
print("\nC) List of devices; list of local_ids:")
print(test_da.ul.ul_sel(device=['Behavior0','Behavior2'], localID=['DOPort0','DOPort1']))
# D) Slice channel by label range (channel must be monotonic)
print("\nD) Slice channel by label range (channel must be monotonic):")
print(test_da.ul.ul_sel(channel=slice('Nosepoke0','Nosepoke6')))

# E) Exact single cell (returns that channel if True, else [])
print("\nE) Exact single cell (returns that channel if True, else []):")
print(test_da.ul.ul_sel(channel='Nosepoke4', device='Behavior1', localID='DOPort1'))

# F) All channel (everything is None -> wildcard)
print("\nF) All channel (everything is None -> wildcard):")
print(test_da.ul.ul_sel(channel=None, device=None, localID=None))

# # G) Slice using numeric index for channel
# print("\nG) Slice using numeric index for channel:")
# print(test_da.ul.ul_sel(channel=channels[0:2]))

# # H) Slice using numeric index for devices
# print("\nH) Slice using numeric index for devices:")
# print(test_da.ul.ul_sel(device=devices[0:2]))

# # I) Slice using numeric index for local_ids
# print("\nI) Slice using numeric index for local_ids:")
# print(test_da.ul.ul_sel(localID=localIDs[0:2]))

# # J) Slice using label range for devices and local_ids
# print("\nJ) Slice using label range for devices and local_ids:")
# print(test_da.ul.ul_sel(device=slice('Behavior1','Behavior2'), localID=slice('DOPort0','DOPort2')))

# # K) Slice using numeric index for devices and local_ids
# print("\nK) Slice using numeric index for devices and local_ids:")
# print(test_da.ul.ul_sel(device=devices[1:3], localID=localIDs[0:3]))

# # L) Slice using label range for channel and blank for devices and local_ids
# print("\nL) Slice using label range for channel and blank for devices and local_ids:")
# print(test_da.ul.ul_sel(channel=slice('Nosepoke0','Nosepoke6')))

# # M) Slice using numeric index for channel and blank for devices and local_ids
# print("\nM) Slice using numeric index for channel and blank for devices and local_ids:")
# print(test_da.ul.ul_sel(channel=channels[2:7], device=devices[:], localID=localIDs[:]))

A) Slice devices by label range, and take the 2nd local_id ('m1'):
['Nosepoke4', 'Nosepoke7']

B) Slice local_ids by label range; any device:
['Nosepoke0', 'Nosepoke1', 'Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5', 'Nosepoke6', 'Nosepoke7', 'Nosepoke8', 'Nosepoke9', 'Nosepoke10', 'Nosepoke11', 'Nosepoke12', 'Nosepoke13', 'Nosepoke14', 'Nosepoke15', 'Nosepoke16', 'Nosepoke17']

C) List of devices; list of local_ids:
['Nosepoke0', 'Nosepoke1', 'Nosepoke6', 'Nosepoke7']

D) Slice channel by label range (channel must be monotonic):
['Nosepoke0', 'Nosepoke1', 'Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5', 'Nosepoke6']

E) Exact single cell (returns that channel if True, else []):
['Nosepoke4']

F) All channel (everything is None -> wildcard):
['Nosepoke0', 'Nosepoke1', 'Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5', 'Nosepoke6', 'Nosepoke7', 'Nosepoke8', 'Nosepoke9', 'Nosepoke10', 'Nosepoke11', 'Nosepoke12', 'Nosepoke13', 'Nosepoke14', 'Nosepoke15', 'Nosepoke16', 'Nosepoke17'

In [ ]:
def update_base_da(data_array: xr.DataArray= None,
              virtual_map_dict: dict= None,
              device_dfs_dict: dict= None,
              virtual_key_map: dict= None,              # Equivalent to channel_type_registerIDs, {source_key: {}}
              source_key: str= None,
                fill_value= None,
                verbose: bool= False
):
    '''
    Update data_array at (Time, unified_coord) positions using: device_dfs_dict[device][reg_address][localID].   

    Steps per unified_coord:
        Map unified_coord -> (device, localID) via virtual_map_dict
        Map localID -> reg_address via virtual_key_map[source_key][localID]
        Pull source column; align Time to data_array.Time (intersection)
        Optional fill within aligned rows; drop NaNs
        Collect columns into a (Time x unified_coord) patch and assign once

    Parameters:
        data_array (xr.DataArray): Existing DataArray to update.
        virtual_map_dict (dict): Dictionary mapping unified coordinates to their reference information.
        device_dfs_dict (dict): Dictionary of devices and their register DataFrames.
        virtual_key_map (dict): Mapping of source_key to localID to register_address.
        source_key (str): The source_key corresponding to the data_array.
        fill_value: Value to use for missing data. If None, will use NaN for floats and False for bools.
        verbose (bool): If True, prints detailed output during the update process.

    Returns:
        xr.DataArray: Updated DataArray with new data.
    '''

    #===== Error Checks =====
    if device_dfs_dict is None:
        raise ValueError("device_dfs_dict is required for this update path.")
    if virtual_key_map is None or source_key is None:
        raise ValueError("virtual_key_map and source_key are required.")
    if data_array is None:
        raise ValueError("data_array is required.")
    if virtual_map_dict is None:
        raise ValueError("virtual_map_dict is required.")
    if not isinstance(data_array, xr.DataArray):
        raise ValueError("data_array must be an xarray DataArray.")
    

    #===== Setup =====

    da= data_array
    da_time= da.get_index('Time')
    # da_unified_coords= da.coords[da.dims[0]].to_index()
    da_unified_coords= da.coords[da.dims[1]].to_index()
    if verbose: print(f'Starting update of data_array with shape {da.shape} using source_key "{source_key}".')
    if verbose: print(f'data_array has {len(da_unified_coords)} unified coordinates, coords are: {da_unified_coords.tolist()}')

    patch_cols= {} # unified_coord -> aligned pandas column

    #===== Main Loop Over Unified Coordinates =====

    for unified_coord, entry in virtual_map_dict.items():

        device, localID= entry                                                   # get device, localID for unified_coord

        reg_addr= virtual_key_map[source_key][localID]                                                   # get reg addr for localID for given source_key

        if device not in list(device_dfs_dict.keys()):
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no device data; skipping.')
            continue
        if reg_addr in list(device_dfs_dict[device].keys()):
            source_col= device_dfs_dict[device][reg_addr][localID]                          # Get source column (pd Series-like) indexed by Time
        else:
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no register address {reg_addr}; skipping.')
            continue
        # source_col= device_dfs_dict[device][reg_addr][localID]  # Get source column (pd Series-like) indexed by Time
        time_index= da_time.intersection(source_col.index)                                                    # Aligns Time by intersection with da.Time
        if len(time_index) == 0:                                                                                # Skip if no overlap in Time
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no overlapping Time with data_array; skipping.')
            continue
        vals= source_col.reindex(time_index)                                                                    # Re-index to aligned Time, keeps only rows with timestamps present in time_index and with same order
        if fill_value is not None:
            vals= vals.fillna(fill_value)                                                                       # Optional fill within aligned rows
        vals= vals.dropna()                                                                                     # Drop NaNs to avoid overwriting with NaN
        if vals.empty:                                                                                          # Skip if no values remain after dropna
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no values after alignment; skipping.')
            continue
        if unified_coord not in da_unified_coords:                                                                        # Skip if unified_coord not in da_unified_coords
            if verbose:
                print(f'Unified Coord {unified_coord} not found in data_array; skipping.')
            continue
        patch_cols[unified_coord]= vals.rename(unified_coord)                                                               # Store aligned column for this unified_coord
    if verbose:
        print(f'Collected {len(patch_cols)} columns for updating data_array.')
    if not patch_cols:
        if verbose:
            print('No columns to update; returning original data_array.')
        return da
    patch_df= pd.DataFrame(patch_cols)                                                                       # Construct patch DataFrame (Time x unified_coord)
    if verbose:
        print(f'Patch DataFrame shape: {patch_df.shape}')
    da.loc[dict(zip(da.dims, [da.coords[da.dims[1]].to_index(), patch_df.index]))]= patch_df.values  # Assign patch to data_array
    if verbose:
        print('Data array updated successfully.')
    return da

In [ ]:
from Refactor.HarpExtender.harp_extender_core import collect_device_dfs

experiment_directory_path= './Bonsai_logs/2025-10-07T18-08-45'
harp_device_yaml_path= './device.yml'


device_dfs_dict= collect_device_dfs(
    experiment_directory_path= experiment_directory_path,
    harp_device_yaml_path= harp_device_yaml_path,
    device_type= device_type,
    device_list= device_list,
    device_IDs= device_IDs,
    device_registers_dict= device_registers_dict,
    flatten= False,
    verbose= True,
    validate= False,
)

device_dfs_dict


--- Processing devices and registers ---

Processing device: Behavior0
  - Loaded register TimestampSeconds (address 8) with shape (1313, 1)
  - Loaded register DigitalInputState (address 32) with shape (8, 4)
  - Loaded register OutputSet (address 34) with shape (424, 14)
  - Loaded register OutputClear (address 35) with shape (447, 14)
  - Loaded register AnalogData (address 44) with shape (1312286, 3)
  - Loaded register StartCameras (address 78) with shape (1, 2)
  - Loaded register Camera0Frame (address 92) with shape (78682, 1)

Processing device: Behavior1
  - Loaded register TimestampSeconds (address 8) with shape (1313, 1)
  - Loaded register DigitalInputState (address 32) with shape (6, 4)
  - Loaded register OutputSet (address 34) with shape (423, 14)
  - Loaded register OutputClear (address 35) with shape (447, 14)
  - Loaded register AnalogData (address 44) with shape (1312389, 3)
  - Loaded register StartCameras (address 78) with shape (1, 2)
  - Loaded register Camera0F

{'Behavior0': {'8':           TimestampSeconds
  Time                      
  104986.0            104986
  104987.0            104987
  104988.0            104988
  104989.0            104989
  104990.0            104990
  ...                    ...
  106294.0            106294
  106295.0            106295
  106296.0            106296
  106297.0            106297
  106298.0            106298
  
  [1313 rows x 1 columns],
  '32':                DIPort0  DIPort1  DIPort2    DI3
  Time                                           
  105633.255072     True    False    False  False
  105633.568576    False    False    False  False
  105673.389504     True    False    False  False
  105673.471680    False    False    False  False
  106035.302304    False     True    False  False
  106035.322752    False    False    False  False
  106043.345664    False    False     True  False
  106043.443744    False    False    False  False,
  '34':                DOPort0  DOPort1  DOPort2  SupplyPort0  Suppl

In [ ]:

# def collect_timestamps_dict(dfs: dict[str, pd.DataFrame | xr.DataArray],
#                             timestamp_name: str | None = None,
#                             verbose: bool = True,
#                             return_type: str = 'list',  # 'dict' or 'list'
#                             ) -> dict[str, pd.Series] | list[pd.Series]:
#     """
#     Extract timestamps from each DataFrame in a dictionary.

#     Parameters:
#         dfs (dict): A dictionary where keys are identifiers and values are pandas DataFrames.
#         timestamp_name (str, optional): The name of the column containing the timestamps. If None, defaults to 'Time'. Note: case-insensitive.
#         verbose (bool, optional): If True, prints warnings and information during processing. Default is True.
#         return_type (str, optional): If 'list', returns a sorted list of all timestamps. If 'dict', returns a dictionary of timestamps per DataFrame. Default is 'list'. If both are needed, call the function twice.

#     Returns:
#         list of all timestamps from each DataFrame in the dictionary, sorted in ascending order.
#         dict where keys are the same as df_dict and values are the timestamps from each DataFrame.
#     """ 

#     if not isinstance(dfs, dict):
#         raise TypeError("Input must be a dictionary of DataFrames.")
    
#     return_type = return_type.lower()

#     if return_type not in ['list', 'dict', 'both']:
#         raise TypeError("return_type must be 'list', 'dict', or 'both'.")

#     if timestamp_name is None:
#         timestamp_name = 'Time'
    
#     if return_type == 'list':
#         all_timestamps= []

#         for key, df in dfs.items():

#             if not isinstance(df, (pd.DataFrame, xr.DataArray)):
#                 raise TypeError(f"Value for key '{key}' is not a pandas DataFrame or xarray DataArray.")
#             if isinstance(df, xr.DataArray):
#                 df = df.to_dataframe().reset_index()
            
#             # if df.index.name.lower() != timestamp_name.lower():
#             #     if verbose: print(f"Timestamp name '{timestamp_name}' does not match DataFrame index name '{df.index.name}' for key '{key}'.")
            
#             all_timestamps.extend(df.index)

#         return sorted(set(all_timestamps))
    
#     elif return_type == 'dict':
#         timestamps_dict = {}
        
#         for key, df in dfs.items():
            
#             if not isinstance(df, (pd.DataFrame, xr.DataArray)):
#                 raise TypeError(f"Value for key '{key}' is not a pandas DataFrame or xarray DataArray.")
#             if isinstance(df, xr.DataArray):
#                 df = df.to_dataframe().reset_index()
            
#             if df.index.name.lower() != timestamp_name.lower():
#                 if verbose: print(f"Timestamp name '{timestamp_name}' does not match DataFrame index name '{df.index.name}' for key '{key}'.")
            
#             timestamps_dict[key] = df.index

#         return timestamps_dict

#     elif return_type == 'both':
#         timestamps_dict = {}
#         all_timestamps= []

#         for key, df in dfs.items():
            
#             if not isinstance(df, (pd.DataFrame, xr.DataArray)):
#                 raise TypeError(f"Value for key '{key}' is not a pandas DataFrame or xarray DataArray.")
#             if isinstance(df, xr.DataArray):
#                 df = df.to_dataframe().reset_index()
            
#             if df.index.name.lower() != timestamp_name.lower():
#                 if verbose: print(f"Timestamp name '{timestamp_name}' does not match DataFrame index name '{df.index.name}' for key '{key}'.")
            
#             timestamps_dict[key] = df.index
#             all_timestamps.extend(df.index)

#         return timestamps_dict, sorted(set(all_timestamps))

In [ ]:
device_dfs_dict.keys()

isinstance(device_dfs_dict['Behavior0'], dict)
channel_type_registerIDs[type_key]
needed_registers= sorted(set(channel_type_registerIDs[type_key].values()))
needed_registers

['34']

In [ ]:
# def collect_timestamps_nested_dict(device_dfs_dict: dict[str, dict[str, pd.DataFrame | xr.DataArray]],
#                                    data_key: str | None = None,
#                                    devices: list[str] | None = None,
#                                    peripheral_registerIDs: list[str] | None = None,
#                                    return_type: str = 'list', # 'dict' or 'list' or 'both'
#                                    verbose: bool = True,
#                                    ) -> dict[str, pd.Series] | list[pd.Series] | tuple[dict[str, pd.Series], list[pd.Series]]:
#     '''
#     Collect timestamps across all devices/registers relevant to the given type_key from the nested device_dfs_dict using collect_timestamps_dict.

#     Parameters:
#         device_dfs_dict (dict): device_dfs_dict {device: {register_address: DataFrame(Time index, localID columns)}}
#         data_key (str): optional peripheral type to use (e.g., the global type_key) if using peripheral_registerIDs
#         devices (list or None): optional subset of devices to include; defaults to all in device_dfs_dict
#         peripheral_registerIDs (list or None): optional subset of peripheral register IDs to include; defaults to all for each device
#         return_type (str): 'list', 'dict', or 'both' (passed through to collect_timestamps_dict)
#         verbose (bool): If True, prints warnings and information during processing.
    
#     Returns:
#         list | dict | (list, dict): timestamps per return_type
#     '''
#     # Guard clause
#     if not isinstance(device_dfs_dict, dict) or not device_dfs_dict:
#         print("Warning: device_dfs_dict is not a valid non-empty dictionary.")
#         return [] if return_type == 'list' else ({} if return_type == 'dict' else ([], {}))
    

#     #=== 1. Obtain Devices
#     devices= devices if devices is not None else list(device_dfs_dict.keys())

#     #=== 2. Filter registers based on devices and peripheral_registerIDs

#     if peripheral_registerIDs is not None and data_key is not None:
#         reg_map= peripheral_registerIDs[data_key]
#         needed_registers= sorted(set(reg_map.values()))

#     else:
#          raise ValueError("Either peripheral_registerIDs and data_key must be provided together. Currently does not support collecting all registers without filtering.")
#     df_dict= {}  # Keyed by "{device}:{register}"

#     for device in devices:
#         if device not in device_dfs_dict:
#             print(f"Warning: Device {device} not found in device_dfs_dict; skipping.")
#             continue

#         dev_regs= device_dfs_dict[device]
#         for reg in needed_registers:
#             if reg not in dev_regs:
#                 print(f"Warning: Device {device} missing register {reg}; skipping.")
#                 continue

#             # Restrict columns to localIDs relevant to this data_key and register
#             localIDs_for_reg= [lid for lid, r in reg_map.items() if r == reg and lid in dev_regs[reg].columns]
#             if not localIDs_for_reg:
#                 print(f"Warning: Device {device} register {reg} has no expected localIDs present; skipping.")
#                 continue

#             df_subset= dev_regs[reg][localIDs_for_reg]
#             df_dict[f"{device}:{reg}"]= df_subset
#     if not df_dict:
#         print("Warning: No DataFrames collected for the specified data_key.")
#         return [] if return_type == 'list' else ({} if return_type == 'dict' else ([], {}))
#     if return_type == 'list':
#         return collect_timestamps_dict(dfs=df_dict, return_type='list')
#     elif return_type == 'dict':
#         return collect_timestamps_dict(dfs=df_dict, return_type='dict')
#     elif return_type == 'both':
#         return collect_timestamps_dict(dfs=df_dict, return_type='both')
#     else:
#         raise ValueError("return_type must be one of 'list', 'dict', or 'both'.")
# times = collect_timestamps_nested_dict(
#     device_dfs_dict=device_dfs_dict,
#     data_key=type_key,
#     peripheral_registerIDs=channel_type_registerIDs,
#     devices=device_list,
#     return_type="list",
#     verbose=True,
# )
# print(type(times), times is None, len(times) if times is not None else None)


<class 'list'> False 818


In [ ]:
# device_dfs_dict.items()
for key, df in device_dfs_dict.items():
    print(f'Device: {key}')
    print(f'type(device_dfs_dict[key])): {type(device_dfs_dict[key])}')
    print(f'type(df): {type(df)}')
     
    print(device_dfs_dict[key].keys())
    for reg_addr, reg_df in device_dfs_dict[key].items():
        print(f'  Register Address: {reg_addr}')
        print(reg_df.index)

Device: Behavior0
type(device_dfs_dict[key])): <class 'dict'>
type(df): <class 'dict'>
dict_keys(['8', '32', '34', '35', '44', '78', '92'])
  Register Address: 8
Index([104986.0, 104987.0, 104988.0, 104989.0, 104990.0, 104991.0, 104992.0,
       104993.0, 104994.0, 104995.0,
       ...
       106289.0, 106290.0, 106291.0, 106292.0, 106293.0, 106294.0, 106295.0,
       106296.0, 106297.0, 106298.0],
      dtype='float64', name='Time', length=1313)
  Register Address: 32
Index([105633.255072, 105633.568576, 105673.389504,  105673.47168,
       106035.302304, 106035.322752, 106043.345664, 106043.443744],
      dtype='float64', name='Time')
  Register Address: 34
Index([104990.222496, 104992.237504,  104996.25248, 104998.269504,
       105003.293504, 105004.306496,  105008.33648, 105008.338496,
       105008.341504,  105011.36848,
       ...
       106268.482496, 106268.485504, 106271.501504, 106275.506496,
       106278.533504, 106281.557504, 106286.575488,  106289.59248,
        106293.6

In [ ]:
# from Refactor.Timestamps.timestamps import collect_timestamps_dict

# def get_df_dict_timestamps(df_dict, return_type='list'):
#     """
#     Extract timestamps from each DataFrame in a dictionary.

#     Parameters:
#         df_dict (dict): A dictionary where keys are identifiers and values are pandas DataFrames.
#         return_type (str, optional): If 'list', returns a sorted list of all timestamps. If 'dict', returns a dictionary of timestamps per DataFrame. Default is 'list'. If both are needed, call the function twice.

#     Returns:
#         list of all timestamps from each DataFrame in the dictionary, sorted in ascending order.
#         dict where keys are the same as df_dict and values are the timestamps from each DataFrame.
#     """
#     if not isinstance(df_dict, dict):
#         raise ValueError("Input must be a dictionary.")
    
#     if return_type is None:
#         raise ValueError("return_type must be specified as 'list' or 'dict'.")
    
#     if return_type == 'list':
#         all_timestamps = []
#         for key, df in df_dict.items():
#             # if not isinstance(df, pd.DataFrame):
#             #     raise ValueError(f"Value for key '{key}' is not a pandas DataFrame.")
#             # if df.index.name != 'Time':
#             #     raise ValueError(f"DataFrame for key '{key}' must have index named 'Time'.")
            
#             all_timestamps.extend(df.index)

#         return sorted(set(all_timestamps))
    
#     elif return_type == 'dict':
#         timestamps_dict = {}
#         for key, df in df_dict.items():
#             # if not isinstance(df, pd.DataFrame):
#             #     raise ValueError(f"Value for key '{key}' is not a pandas DataFrame.")
#             # if df.index.name != 'Time':
#             #     raise ValueError(f"DataFrame for key '{key}' must have index named 'Time'.")
            
#             timestamps_dict[key] = df.index

#         return timestamps_dict
    
#     elif return_type == 'both':
#         all_timestamps = []
#         timestamps_dict = {}
#         for key, df in df_dict.items():
#             if not isinstance(df, pd.DataFrame):
#                 raise ValueError(f"Value for key '{key}' is not a pandas DataFrame.")
#             if df.index.name != 'Time':
#                 raise ValueError(f"DataFrame for key '{key}' must have index named 'Time'.")

#             all_timestamps.extend(df.index)
#             timestamps_dict[key] = df.index

#         return sorted(set(all_timestamps)), timestamps_dict
# times = get_df_dict_timestamps(df_dict=device_dfs_dict, return_type='list')

from Refactor.Timestamps.timestamps import collect_timestamps_nested_dict

times= collect_timestamps_nested_dict(
    device_dfs_dict= device_dfs_dict,
    data_key= type_key,
    peripheral_registerIDs= channel_type_registerIDs,
    devices= device_list,
    return_type= 'list',
    verbose= True
)


print(len(times))
# test construct_channel_type_da
test_da= construct_base_da(
    unified_dim= channel_list,
    unified_coord_name= 'channel',
    virtual_coord_names= ['device', 'localID'],
    virtual_map_dict= channel_ref_dict,
    dict_of= 'tuples',
    times= times,
    name= f'{type_key}_data_array',
    verbose= True

)
print(len(times))

display(test_da.to_dataframe().unstack('channel').drop(columns=['device', 'localID']))

Warning! Timestamp checking is broken
54
No values provided, test_help_values is False. Creating NaNs placeholder DataArray with 54 timestamps and 18 unified coordinates.
54


Activations_data_array                                          \
channel                    Nosepoke0 Nosepoke1 Nosepoke2 Nosepoke3 Nosepoke4   
Time                                                                           
105136.375456                  False     False     False     False     False   
105136.670336                  False     False     False     False     False   
105164.753920                  False     False     False     False     False   
105164.971648                  False     False     False     False     False   
105451.754080                  False     False     False     False     False   
105451.893088                  False     False     False     False     False   
105539.656512                  False     False     False     False     False   
105539.954304                  False     False     False     False     False   
105556.648256                  False     False     False     False     False   
105556.684160                  False     False     False     False     False   
105556.701728                  False     False     False     False     False   
105556.788480                  False     False     False     False     False   
105633.255072                  False     False     False     False     False   
105633.568576                  False     False     False     False     False   
105673.389504                  False     False     False     False     False   
105673.471680                  False     False     False     False     False   
105697.369472                  False     False     False     False     False   
105697.573664                  False     False     False     False     False   
105766.290784                  False     False     False     False     False   
105766.619072                  False     False     False     False     False   
105782.499168                  False     False     False     False     False   
105782.553984                  False     False     False     False     False   
105786.995264                  False     False     False     False     False   
105787.055936                  False     False     False     False     False   
105790.470816                  False     False     False     False     False   
105790.591968                  False     False     False     False     False   
105790.608864                  False     False     False     False     False   
105790.743616                  False     False     False     False     False   
105792.610464                  False     False     False     False     False   
105792.791680                  False     False     False     False     False   
105798.357952                  False     False     False     False     False   
105798.541792                  False     False     False     False     False   
105799.842304                  False     False     False     False     False   
105799.906944                  False     False     False     False     False   
105850.543296                  False     False     False     False     False   
105850.566944                  False     False     False     False     False   
105850.584288                  False     False     False     False     False   
105850.825312                  False     False     False     False     False   
105860.990208                  False     False     False     False     False   
105861.171392                  False     False     False     False     False   
105883.071616                  False     False     False     False     False   
105883.147104                  False     False     False     False     False   
105908.860672                  False     False     False     False     False   
105909.231936                  False     False     False     False     False   
105924.389280                  False     False     False     False     False   
105924.718752                  False     False     False     False     False   
105935.863360                  False     False     False     False     False   
105935.956160 

In [ ]:
def update_base_da(
    data_array: xr.DataArray = None,
    virtual_map_dict: dict = None,
    device_dfs_dict: dict = None,
    virtual_key_map: dict = None,   # {source_key: {localID: reg_address}}
    source_key: str = None,
    fill_value=None,
    verbose: bool = False,
):
    """
    Update data_array at (Time, unified_dim) positions using:
      device_dfs_dict[device][reg_address][localID]
    Robust to dim order changes by inferring dim names.
    """
    # ===== Error Checks =====
    if device_dfs_dict is None:
        raise ValueError("device_dfs_dict is required for this update path.")
    if virtual_key_map is None or source_key is None:
        raise ValueError("virtual_key_map and source_key are required.")
    if data_array is None:
        raise ValueError("data_array is required.")
    if virtual_map_dict is None:
        raise ValueError("virtual_map_dict is required.")
    if not isinstance(data_array, xr.DataArray):
        raise ValueError("data_array must be an xarray DataArray.")

    da = data_array

    # ===== Infer dims safely (no positional indexing) =====
    if "Time" not in da.dims:
        raise ValueError(f"Expected a 'Time' dim, got da.dims={da.dims}")
    time_dim = "Time"

    other_dims = [d for d in da.dims if d != time_dim]
    if len(other_dims) != 1:
        raise ValueError(f"Expected exactly 2 dims (Time + unified). Got da.dims={da.dims}")
    unified_dim = other_dims[0]

    da_time = da.get_index(time_dim)
    da_unified = da.get_index(unified_dim)

    if verbose:
        print(f"Updating DataArray dims={da.dims} (time_dim={time_dim}, unified_dim={unified_dim})")

    patch_cols = {}  # unified_coord -> aligned Series (indexed by Time)

    for unified_coord, entry in virtual_map_dict.items():
        device, localID = entry
        reg_addr = virtual_key_map[source_key][localID]

        if device not in device_dfs_dict:
            continue
        if reg_addr not in device_dfs_dict[device]:
            continue
        if localID not in device_dfs_dict[device][reg_addr].columns:
            continue

        source_col = device_dfs_dict[device][reg_addr][localID]  # Series indexed by Time
        time_index = da_time.intersection(source_col.index)
        if len(time_index) == 0:
            continue

        vals = source_col.reindex(time_index)
        if fill_value is not None:
            vals = vals.fillna(fill_value)
        vals = vals.dropna()
        if vals.empty:
            continue

        if unified_coord not in da_unified:
            continue

        patch_cols[unified_coord] = vals.rename(unified_coord)

    if not patch_cols:
        if verbose:
            print("No columns to update; returning original.")
        return da

    patch_df = pd.DataFrame(patch_cols)  # index=Time, columns=unified

    # ===== Correct assignment (explicit dim names) =====
    da.loc[{time_dim: patch_df.index, unified_dim: patch_df.columns}] = patch_df.to_numpy()
    return da



updated_da= update_base_da(data_array= test_da,
                            virtual_map_dict= channel_ref_dict,
                            device_dfs_dict= device_dfs_dict,
                            virtual_key_map= channel_type_registerIDs,
                            source_key= type_key,
                            fill_value= None,
                            verbose= True

)
display(updated_da.to_dataframe())

Updating DataArray dims=('Time', 'channel') (time_dim=Time, unified_dim=channel)


device  localID  Activations_data_array
Time          channel                                               
105136.375456 Nosepoke0   Behavior0  DIPort0                    True
              Nosepoke1   Behavior0  DIPort1                    True
              Nosepoke2   Behavior0  DIPort2                    True
              Nosepoke3   Behavior1  DIPort0                    True
              Nosepoke4   Behavior1  DIPort1                    True
...                             ...      ...                     ...
106094.554592 Nosepoke13  Behavior4  DIPort1                    True
              Nosepoke14  Behavior4  DIPort2                    True
              Nosepoke15  Behavior5  DIPort0                   False
              Nosepoke16  Behavior5  DIPort1                   False
              Nosepoke17  Behavior5  DIPort2                   False

[972 rows x 3 columns]

In [ ]:
def update_base_da(data_array: xr.DataArray= None,
              virtual_map_dict: dict= None,
              device_dfs_dict: dict= None,
              virtual_key_map: dict= None,              # Equivalent to channel_type_registerIDs, {source_key: {}}
              source_key: str= None,
                fill_value= None,
                verbose: bool= False
):
    '''
    Update data_array at (Time, unified_coord) positions using: device_dfs_dict[device][reg_address][localID].   

    Steps per unified_coord:
        Map unified_coord -> (device, localID) via virtual_map_dict
        Map localID -> reg_address via virtual_key_map[source_key][localID]
        Pull source column; align Time to data_array.Time (intersection)
        Optional fill within aligned rows; drop NaNs
        Collect columns into a (Time x unified_coord) patch and assign once

    Parameters:
        data_array (xr.DataArray): Existing DataArray to update.
        virtual_map_dict (dict): Dictionary mapping unified coordinates to their reference information.
        device_dfs_dict (dict): Dictionary of devices and their register DataFrames.
        virtual_key_map (dict): Mapping of source_key to localID to register_address.
        source_key (str): The source_key corresponding to the data_array.
        fill_value: Value to use for missing data. If None, will use NaN for floats and False for bools.
        verbose (bool): If True, prints detailed output during the update process.

    Returns:
        xr.DataArray: Updated DataArray with new data.
    '''

    #===== Error Checks =====
    if device_dfs_dict is None:
        raise ValueError("device_dfs_dict is required for this update path.")
    if virtual_key_map is None or source_key is None:
        raise ValueError("virtual_key_map and source_key are required.")
    if data_array is None:
        raise ValueError("data_array is required.")
    if virtual_map_dict is None:
        raise ValueError("virtual_map_dict is required.")
    if not isinstance(data_array, xr.DataArray):
        raise ValueError("data_array must be an xarray DataArray.")
    

    #===== Setup =====

    da= data_array
    da_time= da.get_index('Time')
    da_unified_coords= da.coords[da.dims[0]].to_index()

    patch_cols= {} # unified_coord -> aligned pandas column

    #===== Main Loop Over Unified Coordinates =====

    for unified_coord, entry in virtual_map_dict.items():

        device, localID= entry                                                   # get device, localID for unified_coord

        reg_addr= virtual_key_map[source_key][localID]                                                   # get reg addr for localID for given source_key

        if device not in list(device_dfs_dict.keys()):
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no device data; skipping.')
            continue
        if reg_addr in list(device_dfs_dict[device].keys()):
            source_col= device_dfs_dict[device][reg_addr][localID]                          # Get source column (pd Series-like) indexed by Time
        else:
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no register address {reg_addr}; skipping.')
            continue
        # source_col= device_dfs_dict[device][reg_addr][localID]  # Get source column (pd Series-like) indexed by Time
        time_index= da_time.intersection(source_col.index)                                                    # Aligns Time by intersection with da.Time
        if len(time_index) == 0:                                                                                # Skip if no overlap in Time
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no overlapping Time with data_array; skipping.')
            continue
        vals= source_col.reindex(time_index)                                                                    # Re-index to aligned Time, keeps only rows with timestamps present in time_index and with same order
        if fill_value is not None:
            vals= vals.fillna(fill_value)                                                                       # Optional fill within aligned rows
        vals= vals.dropna()                                                                                     # Drop NaNs to avoid overwriting with NaN
        if vals.empty:                                                                                          # Skip if no values remain after dropna
            if verbose:
                print(f'Unified Coord {unified_coord} on device {device} localID {localID} has no values after alignment; skipping.')
            continue
        if unified_coord not in da_unified_coords:                                                                        # Skip if unified_coord not in da_unified_coords
            if verbose:
                print(f'Unified Coord {unified_coord} not found in data_array; skipping.')
            continue
        patch_cols[unified_coord]= vals.rename(unified_coord)                                                               # Store aligned column for this unified_coord
    if verbose:
        print(f'Collected {len(patch_cols)} columns for updating data_array.')
    if not patch_cols:
        if verbose:
            print('No columns to update; returning original data_array.')
        return da
    patch_df= pd.DataFrame(patch_cols)                                                                       # Construct patch DataFrame (Time x unified_coord)
    if verbose:
        print(f'Patch DataFrame shape: {patch_df.shape}')
    da.loc[dict(zip(da.dims, [da.coords[da.dims[0]].to_index(), patch_df.index]))]= patch_df.values  # Assign patch to data_array
    if verbose:
        print('Data array updated successfully.')
    return da


        

    





def update_channel_type_da(
        channel_type_da= None, # Existing DataArray to update
        channel_ref_dict= None,     # {channel: (device, localID)}
        dict_of_devices_registers_dfs_dicts= None, # {device: {register_address: DataFrame(index= Time, columns= localIDs)}}
        device_register_dfs_dict= None, # {device: DataFrame(index= Time, columns= localIDs)}
        channel_type_registerIDs= None, # {type_key: {localID: register_address}}
        type_key= None,
        fill_value= None,
        verbose= False

):
    
    '''
    Update channel_type_da at (Time, channel) positions using: dict_of_devices_registers_dfs_dicts[device][reg_address][localID].   

    Steps per channel:
        Map channel -> (device, localID) via channel_ref_dict
        Map localID -> reg_address via channel_type_registerIDs[type_key][localID]
        Pull source column; align Time to channel_type_da.Time (intersection)
        Optional fill within aligned rows; drop NaNs
        Collect columns into a (Time x channel) patch and assign once

    Parameters:
        channel_type_da (xr.DataArray): Existing DataArray to update.
        channel_ref_dict (dict): Dictionary mapping channel names to their reference information.
        dict_of_devices_registers_dfs_dicts (dict): Nested dictionary of devices and their register DataFrames.
        device_register_dfs_dict (dict or None): Optional pre-constructed dict of device DataFrames to use instead of nested dict.
        channel_type_registerIDs (dict): Mapping of type_key to localID to register_address.
        type_key (str): The type_key corresponding to the channel_type_da.
        fill_value: Value to use for missing data. If None, will use NaN for floats and False for bools.
        verbose (bool): If True, prints detailed output during the update process.

    Returns:
        xr.DataArray: Updated DataArray with new data.
    '''

    #===== Error Checks =====
    if dict_of_devices_registers_dfs_dicts is None:
        raise ValueError("dict_of_devices_registers_dfs_dicts is required for this update path.")
    if channel_type_registerIDs is None or type_key is None:
        raise ValueError("channel_type_registerIDs and type_key are required.")
    if type_key not in channel_type_registerIDs:
        raise ValueError(f"type_key '{type_key}' not found in channel_type_registerIDs.")
    if channel_type_da is None:
        raise ValueError("channel_type_da is required.")
    if channel_ref_dict is None:
        raise ValueError("channel_ref_dict is required.")
    if not isinstance(channel_type_da, xr.DataArray):
        raise ValueError("channel_type_da must be an xarray DataArray.")
    

    #===== Setup =====

    ctda= channel_type_da
    ctda_time= ctda.get_index('Time')
    ctda_channels= ctda.coords['channel'].to_index()

    patch_cols= {} # channel -> aligned pandas column

    #===== Main Loop Over Channels =====

    for channel, (device, localID) in channel_ref_dict.items():

        reg_addr= channel_type_registerIDs[type_key][localID]                                                   # get reg addr for localID for given type_key

        if device not in list(dict_of_devices_registers_dfs_dicts.keys()):
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no device data; skipping.')
            continue

        if reg_addr in list(dict_of_devices_registers_dfs_dicts[device].keys()):
            source_col= dict_of_devices_registers_dfs_dicts[device][reg_addr][localID]                          # Get source column (pd Series-like) indexed by Time
        else:
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no register address {reg_addr}; skipping.')
            continue

        # source_col= dict_of_devices_registers_dfs_dicts[device][reg_addr][localID]  # Get source column (pd Series-like) indexed by Time

        time_index= ctda_time.intersection(source_col.index)                                                    # Aligns Time by intersection with ctda.Time
        if len(time_index) == 0:                                                                                # Skip if no overlap in Time
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no overlapping Time with channel_type_da; skipping.')
            continue

        vals= source_col.reindex(time_index)                                                                    # Re-index to aligned Time, keeps only rows with timestamps present in time_index and with same order
        if fill_value is not None:
            vals= vals.fillna(fill_value)                                                                       # Optional fill within aligned rows
        
        vals= vals.dropna()                                                                                     # Drop NaNs to avoid overwriting with NaN
        if vals.empty:                                                                                          # Skip if no values remain after dropna
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no values after alignment; skipping.')
            continue

        if channel not in ctda_channels:                                                                        # Skip if channel not in ctda_channels
            if verbose:
                print(f'Channel {channel} not found in channel_type_da; skipping.')
            continue

        patch_cols[channel]= vals.rename(channel)                                                               # Store aligned column for this channel

    if not patch_cols:
        if verbose:
            print("No valid data found to update channel_type_da; returning original.")
        return ctda
    
    #===== Construct Patch DataFrame =====
    patch_df= pd.concat(patch_cols.values(), axis=1)                                                            # Combine all columns into a DataFrame indexed by Time
    patch_df= patch_df.reindex(index= ctda_time, 
                               columns= [c for c in ctda_channels if c in patch_df.columns])                    # Reindex to ctda_time and ctda_channels, keeps only channels present in patch_df
    
    if verbose:
        print(f'Constructed patch DataFrame with shape: {patch_df.shape} for updating channel_type_da.')
        display(patch_df)
    #===== Update DataArray =====

    #===== Update DataArray (only write where values exist) =====
    for ch in patch_df.columns:
        col = patch_df[ch].dropna()
        if not col.empty:
            ctda.loc[dict(Time=col.index, channel=ch)] = col.astype(ctda.dtype).values

    if verbose:
        print(f'Updated channel_type_da with data from {len(patch_cols)} channels.')
        print(f'New shape: {ctda.shape}')

    return ctda

In [ ]:
#### 13a) Update Individual Channel Type DataArray Entry

def update_channel_type_da_entry(channel_type_da= None,     # Channel Type DataArray to update. Structure: _.loc[dict(Time= time,
                                                                # channel= channel_type_da.cdl.select_channels_from_C_D_L(channel= None, device= device, localID= localID))]
                                 time= None,                # Time to update. 
                                 channel= None,             # Optional, not sure why this would be used but included for completeness
                                 device= None,              # Device to update.
                                 localID=None,              # LocalID to update.
                                 reg_address=None,          # Register address to update.
                                 dict_of_devices_registers_dfs_dicts= None,  # Nested dict of DataFrames with actual data. Structure: _.[device][register_address][localID].loc(time)
                                 device_register_dfs_dict= None,             # Dict of dataframes for single register address. Structure: _.[device][localID].loc(time)
                                 ):
    '''
    Update a single entry in the channel type DataArray with actual data from the provided DataFrames.
    -----------------------------------
    Parameters:
        channel_type_da (xr.DataArray): DataArray with channel type information to update.
        time: Time to update.
        channel (str): Channel to update.
        device (str): Device to update.
        localID (str): LocalID to update.
        reg_address (str): Register address to update.
        dict_of_devices_registers_dfs_dicts (dict): Nested dict of DataFrames with actual data.
        device_register_dfs_dict (dict): Dict of dataframes for single register address.
    '''
    if time is None:
        time= dict_of_devices_registers_dfs_dicts[device][reg_address][localID].index[0] if dict_of_devices_registers_dfs_dicts is not None else device_register_dfs_dict[device][localID].index[0]

    if reg_address is None and device_register_dfs_dict is not None:
        channel_type_da.loc[dict(Time= time, channel= channel_type_da.cdl.select_channels_from_C_D_L(channel= None, device= device, localID= localID))] = device_register_dfs_dict[device][localID].loc[time]

    if reg_address is not None and device_register_dfs_dict is None:
        channel_type_da.loc[dict(Time= time, channel= channel_type_da.cdl.select_channels_from_C_D_L(channel= None, device= device, localID= localID))] = dict_of_devices_registers_dfs_dicts[device][reg_address][localID].loc[time]

    return None


#### 13b v2) Update Channel Type DataArray

def update_channel_type_da(
        channel_type_da= None, # Existing DataArray to update
        channel_ref_dict= None,     # {channel: (device, localID)}
        dict_of_devices_registers_dfs_dicts= None, # {device: {register_address: DataFrame(index= Time, columns= localIDs)}}
        device_register_dfs_dict= None, # {device: DataFrame(index= Time, columns= localIDs)}
        channel_type_registerIDs= None, # {type_key: {localID: register_address}}
        type_key= None,
        fill_value= None,
        verbose= False

):
    
    '''
    Update channel_type_da at (Time, channel) positions using: dict_of_devices_registers_dfs_dicts[device][reg_address][localID].   

    Steps per channel:
        Map channel -> (device, localID) via channel_ref_dict
        Map localID -> reg_address via channel_type_registerIDs[type_key][localID]
        Pull source column; align Time to channel_type_da.Time (intersection)
        Optional fill within aligned rows; drop NaNs
        Collect columns into a (Time x channel) patch and assign once

    Parameters:
        channel_type_da (xr.DataArray): Existing DataArray to update.
        channel_ref_dict (dict): Dictionary mapping channel names to their reference information.
        dict_of_devices_registers_dfs_dicts (dict): Nested dictionary of devices and their register DataFrames.
        device_register_dfs_dict (dict or None): Optional pre-constructed dict of device DataFrames to use instead of nested dict.
        channel_type_registerIDs (dict): Mapping of type_key to localID to register_address.
        type_key (str): The type_key corresponding to the channel_type_da.
        fill_value: Value to use for missing data. If None, will use NaN for floats and False for bools.
        verbose (bool): If True, prints detailed output during the update process.

    Returns:
        xr.DataArray: Updated DataArray with new data.
    '''

    #===== Error Checks =====
    if dict_of_devices_registers_dfs_dicts is None:
        raise ValueError("dict_of_devices_registers_dfs_dicts is required for this update path.")
    if channel_type_registerIDs is None or type_key is None:
        raise ValueError("channel_type_registerIDs and type_key are required.")
    if type_key not in channel_type_registerIDs:
        raise ValueError(f"type_key '{type_key}' not found in channel_type_registerIDs.")
    if channel_type_da is None:
        raise ValueError("channel_type_da is required.")
    if channel_ref_dict is None:
        raise ValueError("channel_ref_dict is required.")
    if not isinstance(channel_type_da, xr.DataArray):
        raise ValueError("channel_type_da must be an xarray DataArray.")
    

    #===== Setup =====

    ctda= channel_type_da
    ctda_time= ctda.get_index('Time')
    ctda_channels= ctda.coords['channel'].to_index()

    patch_cols= {} # channel -> aligned pandas column

    #===== Main Loop Over Channels =====

    for channel, (device, localID) in channel_ref_dict.items():

        reg_addr= channel_type_registerIDs[type_key][localID]                                                   # get reg addr for localID for given type_key

        if device not in list(dict_of_devices_registers_dfs_dicts.keys()):
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no device data; skipping.')
            continue

        if reg_addr in list(dict_of_devices_registers_dfs_dicts[device].keys()):
            source_col= dict_of_devices_registers_dfs_dicts[device][reg_addr][localID]                          # Get source column (pd Series-like) indexed by Time
        else:
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no register address {reg_addr}; skipping.')
            continue

        # source_col= dict_of_devices_registers_dfs_dicts[device][reg_addr][localID]  # Get source column (pd Series-like) indexed by Time

        time_index= ctda_time.intersection(source_col.index)                                                    # Aligns Time by intersection with ctda.Time
        if len(time_index) == 0:                                                                                # Skip if no overlap in Time
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no overlapping Time with channel_type_da; skipping.')
            continue

        vals= source_col.reindex(time_index)                                                                    # Re-index to aligned Time, keeps only rows with timestamps present in time_index and with same order
        if fill_value is not None:
            vals= vals.fillna(fill_value)                                                                       # Optional fill within aligned rows
        
        vals= vals.dropna()                                                                                     # Drop NaNs to avoid overwriting with NaN
        if vals.empty:                                                                                          # Skip if no values remain after dropna
            if verbose:
                print(f'Channel {channel} on device {device} localID {localID} has no values after alignment; skipping.')
            continue

        if channel not in ctda_channels:                                                                        # Skip if channel not in ctda_channels
            if verbose:
                print(f'Channel {channel} not found in channel_type_da; skipping.')
            continue

        patch_cols[channel]= vals.rename(channel)                                                               # Store aligned column for this channel

    if not patch_cols:
        if verbose:
            print("No valid data found to update channel_type_da; returning original.")
        return ctda
    
    #===== Construct Patch DataFrame =====
    patch_df= pd.concat(patch_cols.values(), axis=1)                                                            # Combine all columns into a DataFrame indexed by Time
    patch_df= patch_df.reindex(index= ctda_time, 
                               columns= [c for c in ctda_channels if c in patch_df.columns])                    # Reindex to ctda_time and ctda_channels, keeps only channels present in patch_df
    
    if verbose:
        print(f'Constructed patch DataFrame with shape: {patch_df.shape} for updating channel_type_da.')
        display(patch_df)
    #===== Update DataArray =====

    #===== Update DataArray (only write where values exist) =====
    for ch in patch_df.columns:
        col = patch_df[ch].dropna()
        if not col.empty:
            ctda.loc[dict(Time=col.index, channel=ch)] = col.astype(ctda.dtype).values

    if verbose:
        print(f'Updated channel_type_da with data from {len(patch_cols)} channels.')
        print(f'New shape: {ctda.shape}')

    return ctda


In [ ]:
@xr.register_dataarray_accessor('ulookup')
class BaseULookupAccessor:
    '''
    Base Accessor for uLookup functionality on xarray DataArrays. Requires lookup array to be pre-constructed. See "LookupAccessorConstructor" for automatic construction and custom accessor creation.

    
    '''

    def __init__(self, da: xr.DataArray):
        self._da= da

    def sel(
            self,
            lookup_array: xr.DataArray,
            *,
            unified_coord_name: str | None = None,
            verbose: bool = False,
            **sel_kwargs,
        ) -> xr.DataArray:
            # Infer unified dim name from lookup if not given
            if unified_coord_name is None:
                unified_coord_name = lookup_array.dims[0]

            # --- selectors that apply to lookup (virtual dims)
            lookup_dims = [d for d in lookup_array.dims if d != unified_coord_name]
            lookup_selectors = {k: v for k, v in sel_kwargs.items() if k in lookup_dims and v is not None}

            # Compute valid unified labels using your existing function
            selected_unified = ulookup(
                lookup_array=lookup_array,
                unified_coord_name=unified_coord_name,
                verbose=verbose,
                **lookup_selectors,
            )

            # --- selectors that apply to the data (real dims)
            data_selectors = {k: v for k, v in sel_kwargs.items() if k in self._da.dims and v is not None}

            # Select on the data: unified dim + any other dim selections
            indexers = {unified_coord_name: selected_unified}
            indexers.update(data_selectors)

            return self._da.sel(**indexers)

    __call__ = sel

        

/tmp/ipykernel_2750805/3837561625.py:1: AccessorRegistrationWarning: registration of accessor <class '__main__.BaseULookupAccessor'> under name 'ulookup' for type <class 'xarray.core.dataarray.DataArray'> is overriding a preexisting attribute with the same name.
  @xr.register_dataarray_accessor('ulookup')


In [ ]:
@xr.register_dataarray_accessor("ulookup")
class BaseULookupAccessor:
    def __init__(self, da: xr.DataArray):
        self._da = da

    def accessor_ulookup(
        self,
        lookup_array: xr.DataArray,
        unified_coord_name: str | None = None,
        verbose: bool = False,
        **selectors,
    ) -> list:
        if unified_coord_name is None:
            unified_coord_name = lookup_array.dims[0]

        return ulookup(
            lookup_arr=lookup_array,          # ✅ use the passed lookup
            unified_coord_name=unified_coord_name,
            verbose=verbose,
            **selectors,
        )

    def sel(
        self,
        lookup_array: xr.DataArray,          # ✅ positional/keyword, but it's its own parameter
        *,
        unified_coord_name: str | None = None,
        verbose: bool = False,
        **sel_kwargs,
    ) -> xr.DataArray:
        if unified_coord_name is None:
            unified_coord_name = lookup_array.dims[0]

        # selectors for lookup dims
        lookup_dims = [d for d in lookup_array.dims if d != unified_coord_name]
        lookup_selectors = {k: v for k, v in sel_kwargs.items() if k in lookup_dims and v is not None}

        selected_unified = self.accessor_ulookup(
            lookup_array,                    # ✅ pass once, positionally
            unified_coord_name=unified_coord_name,
            verbose=verbose,
            **lookup_selectors,
        )

        # selectors for real data dims
        data_selectors = {k: v for k, v in sel_kwargs.items()
                          if k in self._da.dims and k != unified_coord_name and v is not None}

        indexers = {unified_coord_name: selected_unified}
        indexers.update(data_selectors)
        return self._da.sel(**indexers)

    __call__ = sel


/tmp/ipykernel_2750805/2395491642.py:1: AccessorRegistrationWarning: registration of accessor <class '__main__.BaseULookupAccessor'> under name 'ulookup' for type <class 'xarray.core.dataarray.DataArray'> is overriding a preexisting attribute with the same name.
  @xr.register_dataarray_accessor("ulookup")


In [ ]:
da_x = test_da.ulookup.sel(channel_lookup_da, device="Behavior1", localID="DOPort1")

print(da_x)

TypeError: <__main__.base_ulookup_accessor object at 0x72ae61759990> got multiple values for keyword argument 'lookup_arr'

In [ ]:
@xr.register_dataarray_accessor('ulookup')
class base_ulookup_accessor:
    '''
    Generalised lookup accessor, adds methods to xarray DataArrays to select global coordiantes from custom virtual coordinate systems
    ----------------------------------

    '''

    def __init__(self,
                 data_array: xr.DataArray= None,
                 lookup_arr: xr.DataArray | None = None,
                 unified_dim: list= None,
                 unified_coord_name: str= 'unified_coord',
                 virtual_coord_names: list= None,
                 virtual_map_dict: dict= None,
                 dict_of: str= 'dicts',
                 name: str= 'lookup_matrix',
                 verbose: bool= False,

                 ):
        
        if data_array is None: raise ValueError("data_array must be provided.")
        self._da= data_array

        if lookup_arr is None:
            if unified_dim is None or unified_coord_name is None or virtual_coord_names is None or virtual_map_dict is None:
                raise ValueError("If lookup_arr is not provided, unified_dim, unified_coord_name, virtual_coord_names, and virtual_map_dict must be provided to construct the lookup array.")
            self._lookup_da, self._lookup_virtual_coords= construct_lookup_array(
                unified_dim= unified_dim,
                unified_coord_name= unified_coord_name,
                virtual_coord_names= virtual_coord_names,
                virtual_map_dict= virtual_map_dict,
                dict_of= dict_of,
                name= name,
                verbose= verbose
            )
        else:
            self._lookup_da= lookup_arr
        
    def accessor_ulookup(self,
                                unified_coord_name: str= 'unified_coord',
                                verbose: bool= False,
                                **selectors,
                            ) -> list:
            '''
            uLookup: Selects unified/global coordinates from a lookup DataArray based on specified criteria across virtual coordinate dimensions. Supports both positional- and label-based indexing and combinations thereof.
            -----------------------------------
            Parameters:
                unified_coord_name (str): Name of the unified/global coordinate dimension.
                verbose (bool): If True, prints additional information.
                **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by. 
            Returns:
                list: A list of unified/global coordinate names that match the specified criteria.
            '''
            return ulookup(
                lookup_arr= self._lookup_da,
                unified_coord_name= unified_coord_name,
                verbose= verbose,
                **selectors
            )
        

    def sel(self, **sel_kwargs) -> xr.DataArray:
        """
        Wrapper around xarray .sel but using ulookup to choose valid global coordinates.
        """
        # Infer unified coord name from lookup DataArray (first dim by construction)
        unified_coord_name = self._lookup_da.dims[0]

        # Extract selector kwargs for virtual coords in the lookup (exclude unified coord itself)
        lookup_dims = [d for d in self._lookup_da.dims if d != unified_coord_name]
        verbose = sel_kwargs.pop("verbose", False)
        selectors = {k: v for k, v in sel_kwargs.items() if k in lookup_dims and v is not None}

        # Compute valid unified coordinates via lookup
        selected_unified = self.accessor_ulookup(
            lookup_arr=self._lookup_da,
            unified_coord_name=unified_coord_name,
            verbose=verbose,
            **selectors,
        )

        # Build .sel indexers for the target DataArray:
        # - select along the unified coord
        # - pass through any other dimension selectors (e.g., Time)
        da_sel_kwargs = {k: v for k, v in sel_kwargs.items() if k in self._da.dims and k != unified_coord_name}
        indexers = {unified_coord_name: selected_unified}
        indexers.update(da_sel_kwargs)

        return self._da.sel(**indexers)
    
    __call__= sel
        
            
        
    

    

In [ ]:
# Demo custom class using base_ulookup_accessor

@xr.register_dataarray_accessor('channel_lookup')
class channel_lookup_accessor(base_ulookup_accessor):
        def __init__(self,
                 data_array: xr.DataArray= ,
                 lookup_arr: xr.DataArray | None = None,
                 unified_dim: list= None,
                 unified_coord_name: str= 'unified_coord',
                 virtual_coord_names: list= None,
                 virtual_map_dict: dict= None,
                 dict_of: str= 'dicts',
                 name: str= 'lookup_matrix',
                 verbose: bool= False,
                 ):
        

SyntaxError: expected default value expression (1677010436.py, line 6)

In [ ]:
# test_da.dims

ucn= [d for d in test_da.dims if d != 'Time'][0]  # unified coord name
ucn

'channel'

In [ ]:

ucn= [d for d in test_da.dims][1]  # unified coord name
ucn

'channel'

In [ ]:


l= list(test_da.coords.keys())
for coord in l: print(coord) if coord != 'Time' else None 



channel
device
localID


In [ ]:
# Example usage of base_ulookup_accessor

from Refactor.LookupArrays.lookup_arrays import ulookup


base_ulookup_accessor(test_da,
                       lookup_arr= channel_lookup_da,
                       unified_dim= channel_list,
                       unified_coord_name= 'channel',
                       virtual_coord_names= ['device', 'localID'],
                       virtual_map_dict= channel_ref_dict,
                       dict_of= 'tuples',
                       name= 'channel_lookup_matrix',
                       verbose= False
                       )

# Use the ulookup accessor which derives the lookup from the DataArray itself
ulookup= base_ulookup_accessor(
    test_da,
    lookup_arr= channel_lookup_da,
    unified_dim= channel_list,
    unified_coord_name= 'channel',
    virtual_coord_names= ['device', 'localID'],
    virtual_map_dict= channel_ref_dict,
    dict_of= 'tuples',
    name= 'channel_lookup_matrix',
    verbose= False
)

# Select using the pre-built accessor instance to avoid constructor validation
ulookup.sel(device='Behavior2', localID='DOPort1')

TypeError: <__main__.base_ulookup_accessor object at 0x72ae61759990> got multiple values for keyword argument 'lookup_arr'

In [ ]:
@xr.register_dataarray_accessor('ulookup')
class ULookupAccessor:
    '''
    Generalised lookup accessor, adds methods to xarray DataArrays to select global coordiantes from custom virtual coordinate systems
    -----------------------------------

    Assumptions:
        - unified_coord_name is dimension in the DataArray
        - each virtual coordinate is 1D coordinate with dims == (unified_coord_name,)

    '''

    def __init__(self,
                 data_array: xr.DataArray
                 ):
        self._da= data_array
        self._lookup_cache: dict[tuple, xr.DataArray]= {}  # Cache for lookup DataArrays keyed by (unified_coord_name, virtual_coord_names tuple)

    #==== Infer 
    

    

        



        
        

In [ ]:
@xr.register_dataarray_accessor('cdl')
class CDLAccessor:
    """
    This accessor adds methods to xarray DataArrays to select channels using a
    channel-device-localID lookup matrix.
    """

    #===== Constructor =====
    def __init__(self, da: xr.DataArray):
        self._da = da

        # Checks
        if "channel" not in self._da.dims:
            raise ValueError("DataArray must have 'channel' as one of its dimensions.")
        for coord in ["device", "localID"]:
            if coord not in self._da.coords:
                raise ValueError(f"DataArray must have '{coord}' as one of its coordinates.")

        #--- Create lookup DataArray
        channels = da.channel.values.tolist()

        devices = list(dict.fromkeys(da.device.values.tolist()))
        localIDs = list(dict.fromkeys(da.localID.values.tolist()))

        lookup_arr = np.zeros((len(channels), len(devices), len(localIDs)), dtype=bool)

        for i, (d, l) in enumerate(zip(da.device.values, da.localID.values)):
            device_index = devices.index(d)
            localID_index = localIDs.index(l)
            lookup_arr[i, device_index, localID_index] = True

        self.lookup_da = xr.DataArray(
            data=lookup_arr,
            dims=["channel", "device", "localID"],
            coords={
                "channel": channels,
                "device": devices,
                "localID": localIDs,
            },
            name="channel_lookup_matrix",
            attrs={
                "description": "Channel lookup matrix indicating presence of channels across devices and local IDs",
                "source": "Constructed using CDLAccessor",
            },
        )

    #===== Methods =====
    def select_channels_from_C_D_L(self, channel=None, device=None, localID=None):
        """
        Select channels from the Channel-Device-LocalID lookup DataArray.
        """

        sel = {}
        if channel is not None:
            sel["channel"] = channel
        if device is not None:
            sel["device"] = device
        if localID is not None:
            sel["localID"] = localID

        # Subset
        subset = self.lookup_da.sel(**sel) if sel else self.lookup_da
        print(f"Subset shape after selection: {subset.shape}")

        # Reduce across non-channel dimensions
        other_dims = [d for d in subset.dims if d != "channel"]
        if other_dims:
            per_channel_true = subset.any(dim=other_dims)
        else:
            per_channel_true = subset.astype(bool)

        print(f"Per-channel true shape after reduction: {per_channel_true.shape}")

        return per_channel_true["channel"].values[per_channel_true.values].tolist()

    def sel(self, *, channel=None, device=None, localID=None, **sel_kwargs):
        """
        Wrapper around xarray .sel but using CDL lookup to choose valid channels.
        """
        selected_channels = self.select_channels_from_C_D_L(
            channel=channel, device=device, localID=localID
        )
        return self._da.sel(channel=selected_channels, **sel_kwargs)

    # Make callable equivalent to .sel
    __call__ = sel

In [ ]:
def select_channels_from_C_D_L(lookup_da: xr.DataArray= None,
                               unified_coord_name: str= 'channel',
                               verbose: bool= False,
                               channel: str= None,
                               device: str= None,
                               localID: str= None) -> list:
    '''
    Select channels from a Channel-Device-LocalID lookup DataArray based on specified criteria. Deprecated: use ulookup instead.
    -----------------------------------
    Parameters:
        lookup_da (xr.DataArray): The Channel-Device-LocalID lookup DataArray.
        unified_coord_name (str): Name of the unified/global coordinate dimension (default is 'channel').
        verbose (bool): If True, prints additional information.
        channel (str): The channel name to filter by (optional).
        device (str): The device name to filter by (optional).
        localID (str): The local ID to filter by (optional).
    '''
    if verbose: print(f'Warning: select_channels_from_C_D_L is deprecated. Use ulookup instead with appropriate parameters.')
    return ulookup(
        lookup_arr= lookup_da,
        unified_coord_name= unified_coord_name,
        verbose= verbose,
        channel= channel,
        device= device,
        localID= localID
    )

 


def select_channels_from_C_D_L(lookup_da= None, channel= None, device= None, localID= None):
    '''
    Select channels from a Channel-Device-LocalID lookup DataArray based on specified criteria.
    -----------------------------------

    Parameters:
        lookup_da (xr.DataArray): The Channel-Device-LocalID lookup DataArray.
        channel (str): The channel name to filter by (optional).
        device (str): The device name to filter by (optional).
        localID (str): The local ID to filter by (optional).

    Returns:
        list: A list of channel names that match the specified criteria.
    '''

    sel= {}

    if channel is not None: sel['channel']= channel
    if device is not None: sel['device']= device
    if localID is not None: sel['localID']= localID

    # Subset
    subset= lookup_da.sel(**sel) if sel else lookup_da
    print(f'Subset shape after selection: {subset.shape}')

    # Reduce across non-channel dimensions
    other_dims= [d for d in subset.dims if d != 'channel']
    if other_dims:
        per_channel_true= subset.any(dim= other_dims)
    else:
        per_channel_true= subset.astype(bool)

    print(f'Per-channel true shape after reduction: {per_channel_true.shape}')

    return per_channel_true['channel'].values[per_channel_true.values].tolist()

In [ ]:
def ulookup(lookup_arr: xr.DataArray = None,
            unified_coord_name: str= 'unified_coord',
            verbose: bool= False,
            **selectors,
        ) -> list:
    '''
    uLookup: Selects unified/global coordinates from a lookup DataArray based on specified criteria across virtual coordinate dimensions. Supports both positional- and label-based indexing and combinations thereof.
    -----------------------------------

    Parameters:
        lookup_arr (xr.DataArray): N-dim Boolean lookup tensor with dimensions like [unified_coord_name] + virtual coordinate dimensions.
        unified_coord_name (str): Name of the unified/global coordinate dimension.
        verbose (bool): If True, prints additional information.
        **selectors: Keyword arguments specifying selection criteria for both real and virtual coordinate dimensions. Keys are  coordinate names, values are the desired values to filter by. 
    
    Returns:
        list: A list of unified/global coordinate names that match the specified criteria.
    '''
    if lookup_arr is None:
        raise ValueError("lookup_arr must be provided.")
    if unified_coord_name not in lookup_arr.dims:
        raise ValueError(f"{unified_coord_name} is not a dimension in the provided lookup_arr.")
    if verbose: print(f'Starting uLookup with unified_coord_name="{unified_coord_name}" and selectors: {selectors}')


    #=== Gather selection criteria
    sel= {}

    sel= {key: value for key, value in selectors.items() if value is not None}
    if verbose: print(f'Selecting unified coordinates with criteria: {sel}')

    #=== Get subset based on selection criteria
    subset= lookup_arr.sel(**sel) if sel else lookup_arr
    if verbose: print(f'Subset shape after selection: {subset.shape}')

    #=== Reduce across non-unified_coord_name dimensions
    other_dims= [d for d in subset.dims if d != unified_coord_name]
    
    if other_dims:
        per_unified_true= subset.any(dim= other_dims)
    else:
        per_unified_true= subset.astype(bool)
    
    if verbose: print(f'Per-unified_coord_name true shape after reduction: {per_unified_true.shape}')
    
    # return per_unified_true[unified_coord_name].values[per_unified_true.values].tolist()

    labels = per_unified_true[unified_coord_name].values
    mask = np.asarray(per_unified_true.values, dtype=bool)
    return labels[mask].tolist()


        



def select_channels_from_C_D_L(lookup_da= None, channel= None, device= None, localID= None):
    '''
    Select channels from a Channel-Device-LocalID lookup DataArray based on specified criteria.
    -----------------------------------

    Parameters:
        lookup_da (xr.DataArray): The Channel-Device-LocalID lookup DataArray.
        channel (str): The channel name to filter by (optional).
        device (str): The device name to filter by (optional).
        localID (str): The local ID to filter by (optional).

    Returns:
        list: A list of channel names that match the specified criteria.
    '''

    sel= {}

    if channel is not None: sel['channel']= channel
    if device is not None: sel['device']= device
    if localID is not None: sel['localID']= localID

    # Subset
    subset= lookup_da.sel(**sel) if sel else lookup_da
    print(f'Subset shape after selection: {subset.shape}')

    # Reduce across non-channel dimensions
    other_dims= [d for d in subset.dims if d != 'channel']
    if other_dims:
        per_channel_true= subset.any(dim= other_dims)
    else:
        per_channel_true= subset.astype(bool)

    print(f'Per-channel true shape after reduction: {per_channel_true.shape}')

    return per_channel_true['channel'].values[per_channel_true.values].tolist()

In [ ]:
def construct_lookup_array(unified_dim: list= None,
                           unified_coord_name: str= 'unified_coord',
                           virtual_coord_names: list= None,
                           virtual_map_dict: dict= None,
                           dict_of: str= 'dicts',
                           name: str= 'lookup_matrix',
                           verbose: bool= False
                           ) -> tuple[xr.DataArray, dict[str, list]]:
    '''
    Construct a Lookup Array DataArray for a set of given unified/global coordinates and their associated virtual coordinates.
    -----------------------------------

    Parameters:
        unified_dim (list): List of unified/global coordinate names to include in the DataArray.
        unified_coord_name (str): Name for the unified/global coordinate dimension.
        virtual_coord_names (list): List of virtual coordinate dimensions to include in the DataArray.
        virtual_map_dict (dict): Dictionary mapping unified/global coordinates to their virtual coordinate information.
        dict_of (str): Specifies the format of virtual_map_dict. Options are 'dicts' or 'tuples'.
        name (str): Name of the DataArray.
        verbose (bool): If True, prints additional information. 
    
    Returns:
        lookup_da (xr.DataArray): A DataArray representing the lookup matrix.
        lookup_virtual_coords (dict): A dictionary containing lists of unique virtual coordinate values for each virtual coordinate dimension. Keys are virtual coordinate names, values are lists of unique values. 
    '''
    #==== Input validation and preprocessing ====
    dict_of= dict_of.lower()

    if dict_of not in ("dicts", "tuples"):
        raise ValueError(f"dict_of must be either 'dicts' or 'tuples', got: {dict_of}")
    
    if unified_dim is None or virtual_coord_names is None or virtual_map_dict is None:
        raise ValueError("unified_dim, virtual_coord_names, and virtual_map_dict must be provided.")
    
    if len(unified_dim) == 0:
        raise ValueError("unified_dim cannot be empty.")
    if len(virtual_coord_names) == 0:
        raise ValueError("virtual_coord_names cannot be empty.")
    
    if verbose: print(f'Constructing lookup array for {len(unified_dim)} unified coordinates and virtual coordinates: {virtual_coord_names}')
    
    missing= [u for u in unified_dim if u not in virtual_map_dict]
    if missing: raise KeyError(f'Error: virtual_map_dict is missing entries for unified coordinates: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    
    if dict_of == "tuples" and any(isinstance(v, dict) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='tuples' but at least one entry is a dict. Use dict_of='dicts'.")

    if dict_of == "dicts" and any(isinstance(v, (tuple, list)) for v in virtual_map_dict.values()):
        raise TypeError("dict_of='dicts' but at least one entry is a tuple/list. Use dict_of='tuples'.")

    #==== Collect Unique Values for Each Virtual Coordinate (Levels)
    lookup_virtual_coords= {vcoord: [] for vcoord in virtual_coord_names}

    for unified in unified_dim:
        virtual_coords= virtual_map_dict[unified]

        if dict_of == 'tuples':
            if not isinstance(virtual_coords, (tuple, list)):
                raise TypeError(f'Expected virtual_map_dict to be dict-of-tuples/lists when dict_of="tuples", but got {type(virtual_coords)} for unified coordinate {unified}')
            if len(virtual_coords) != len(virtual_coord_names):
                raise ValueError(f"Tuple length mismatch for {unified}: len(entry)={len(virtual_coords)} vs len(virtual_coord_names)={len(virtual_coord_names)}")
            
            for vcoord_name, vcoord_value in zip(virtual_coord_names, virtual_coords):
                if vcoord_value not in lookup_virtual_coords[vcoord_name]:
                    lookup_virtual_coords[vcoord_name].append(vcoord_value)

        elif dict_of == 'dicts':
            if not isinstance(virtual_coords, Mapping):
                raise TypeError(f'Expected virtual_map_dict to be dict-of-dicts when dict_of="dicts", but got {type(virtual_coords)} for unified coordinate {unified}')
            
            for vcoord_name in virtual_coord_names:
                if vcoord_name not in virtual_coords:
                    raise KeyError(f'Missing key {vcoord_name} in virtual_map_dict entry for unified coordinate {unified} (dict_of="dicts")')
                
                vcoord_value= virtual_coords[vcoord_name]
                if vcoord_value not in lookup_virtual_coords[vcoord_name]:
                    lookup_virtual_coords[vcoord_name].append(vcoord_value)
        
        else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

        if verbose: 
            for vcoord_name in virtual_coord_names: 
                print(f'Collected {vcoord_name}: {lookup_virtual_coords[vcoord_name]} unique values.')
        

    #==== Construct Lookup Array ====
    
    # Fast lookup tables from virtual coordinate values to integer indices along each virtual dimension
    index_map= {
        vcoord: {val: idx for idx, val in enumerate(lookup_virtual_coords[vcoord])} 
        for vcoord in virtual_coord_names
        }

    # Allocate boolean array for lookup matrix
    lookup_shape= [len(unified_dim)] + [len(lookup_virtual_coords[vcoord]) for vcoord in virtual_coord_names]
    lookup_arr= np.zeros(lookup_shape, dtype= bool)

    # Populate lookup array
    for i, unified in enumerate(unified_dim):
        
        entry= virtual_map_dict[unified]

        if dict_of == 'tuples':
            idx= [i] + [index_map[vcoord_name][vcoord_value] 
            for vcoord_name, vcoord_value in zip(virtual_coord_names, entry)
            ]
        
        elif dict_of == 'dicts':
            idx= [i] + [index_map[vcoord_name][entry[vcoord_name]]
            for vcoord_name in virtual_coord_names
            ]
        else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")
        lookup_arr[tuple(idx)] = True
    
    #==== Create DataArray 
    dims= [unified_coord_name] + virtual_coord_names
    
    coords= {unified_coord_name: unified_dim,}

    coords.update({vcoord: lookup_virtual_coords[vcoord] for vcoord in virtual_coord_names})

    lookup_da= xr.DataArray(
        data= lookup_arr,
        dims= dims,
        coords= coords,
        name= name,
        attrs= {
            'description': 'Lookup matrix indicating presence of unified coordinates across virtual coordinate dimensions',
            'source': 'Constructed using construct_lookup_array function'
        }
    )
    return lookup_da, lookup_virtual_coords






        




def construct_channel_lookup_da(channels= None, channels_ref_dict= None):
    '''
    Construct a Channel Lookup DataArray for a set of given channels.
    -----------------------------------

    Parameters:
        channels (list): List of channel names to include in the DataArray.
        channels_ref_dict (dict): Dictionary mapping channel names to their reference information.

    Returns:
        lookup_da (xr.DataArray): A DataArray representing the channel lookup matrix.
        lookup_devices (list): List of unique devices found in the channels_ref_dict.
        lookup_localIDs (list): List of unique local IDs found in the channels_ref_dict.
    '''

    lookup_devices= []
    lookup_localIDs= []

    for d, l in channels_ref_dict.values():
        if d not in lookup_devices:
            lookup_devices.append(d)
        if l not in lookup_localIDs:
            lookup_localIDs.append(l)

    lookup_arr= np.zeros((len(channels), len(lookup_devices), len(lookup_localIDs)), dtype= bool)

    for i, channel in enumerate(channels):
        look_device, look_localID = channels_ref_dict[channel]
        
        device_index= lookup_devices.index(look_device)
        localID_index= lookup_localIDs.index(look_localID)

        lookup_arr[i, device_index, localID_index] = True

    lookup_da= xr.DataArray(
        data= lookup_arr,
        dims= ['channel', 'device', 'localID'],
        coords= {
            'channel': channels,
            'device': lookup_devices,
            'localID': lookup_localIDs
        },
        name= 'Channel_lookup_matrix',
        attrs= {
            'description': 'Channel lookup matrix indicating presence of channels across devices and local IDs',
            'source': 'Constructed using construct_channel_lookup_da function'
        }
    )
    return lookup_da, lookup_devices, lookup_localIDs

In [ ]:
# def construct_base_da(values: list | np.ndarray= None,
#                       unified_dim: list = None,
#                       unified_coord_name: str= 'unified_coord',
#                       virtual_coord_names: list= None,
#                       virtual_map_dict: dict= None,
#                       dict_of: str= 'dicts',
#                       times: list | np.ndarray= None,
#                       name: str= 'base_dataarray',
#                       test_help_values: bool= False,
#                       verbose: bool= False
#                       ) -> xr.DataArray:
#     '''
#     Construct a base/placeholder DataArray for a set of given unified/global coordinates for all timepoints and their associated virtual coordinates.
#     -----------------------------------

#     Parameters:
#         values (list or np.ndarray): Values to populate the DataArray. If None, a placeholder array of NaNs will be created.
#         unified_dim (list): List of unified/global coordinate names to include in the DataArray.
#         virtual_coord_names (list): List of virtual coordinate dimensions to include in the DataArray.
#         virtual_map_dict (dict): Dictionary mapping unified/global coordinates to their virtual coordinate information.
#         dict_of (str): Specifies the format of virtual_map_dict. Options are 'dicts' or 'tuples'.
#         times (list): List of timepoints to include in the DataArray. This should be obtained using collect_timestamps_dict with return_type='list'.
#         name (str): Name of the DataArray.
#         test_help_values (bool): If True, use test helper values instead of the main values.
#         verbose (bool): If True, print verbose output.
    
#     Returns:
#         A DataArray populated with the specified values, or a placeholder array if no values are provided.
#     '''
    
#     from collections.abc import Mapping

#     dict_of= dict_of.lower()

#     if dict_of not in ("dicts", "tuples"):
#         raise ValueError(f"dict_of must be either 'dicts' or 'tuples', got: {dict_of}")

#     if times is None or unified_dim is None or virtual_coord_names is None or virtual_map_dict is None:
#         raise ValueError("times, unified_dim, virtual_coord_names, and virtual_map_dict must be provided.")
    
#     missing= [u for u in unified_dim if u not in virtual_map_dict]
#     if missing: raise KeyError(f'Error: virtual_map_dict is missing entries for unified coordinates: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    
#     if dict_of == "tuples" and any(isinstance(v, dict) for v in virtual_map_dict.values()):
#         raise TypeError("dict_of='tuples' but at least one entry is a dict. Use dict_of='dicts'.")

#     if dict_of == "dicts" and any(isinstance(v, (tuple, list)) for v in virtual_map_dict.values()):
#         raise TypeError("dict_of='dicts' but at least one entry is a tuple/list. Use dict_of='tuples'.")


#     #=== Create data to populate DataArray 
#     if values is None and test_help_values is False:
#         if verbose: print(f'No values provided, test_help_values is False. Creating NaNs placeholder DataArray with {len(times)} timestamps and {len(unified_dim)} unified coordinates.')
#         values= np.zeros((len(times), len(unified_dim)), dtype= bool) 

#     #=== Create test helper values if specified
#     if values is None and test_help_values is True:
#         if verbose: print(f'No values provided, but test_help_values is True. Creating test helper DataArray with {len(times)} timestamps and {len(unified_dim)} unified coordinates.')
#         value_list = []
#         for _ in times:
#             time_values = []
#             for unified in unified_dim:
#                 virtual_coords = virtual_map_dict[unified]

#                 if dict_of == 'dicts':
#                     if not isinstance(virtual_coords, Mapping): raise TypeError(f'Expected virtual_map_dict to be dict-of-dicts when dict_of="dicts", but got {type(virtual_coords)} for unified coordinate {unified}')
                    
#                     coord_str = ', '.join([f'{k}: {v}' for k, v in virtual_coords.items()])

#                 elif dict_of == 'tuples':
                    
#                     if not isinstance(virtual_coords, (tuple, list)): raise TypeError(f'Expected virtual_map_dict to be dict-of-tuples/lists when dict_of="tuples", but got {type(virtual_coords)} for unified coordinate {unified}')
                    
#                     if len(virtual_coords) != len(virtual_coord_names): raise ValueError(f"Tuple length mismatch for {unified}: len(entry)={len(virtual_coords)} vs len(virtual_coord_names)={len(virtual_coord_names)}")
                    
#                     coord_str = ", ".join([f"{k}: {v}" for k, v in zip(virtual_coord_names, virtual_coords)])

#                 else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

#                 time_values.append(f'({unified}, {coord_str})')

#             value_list.append(time_values)
#         values = np.array(value_list)
#         if verbose: print(f'Constructed test helper values array with shape: {values.shape}')
    
#     if values is not None and test_help_values is True:
#         print('Warning: Both values provided and test_help_values is True. Using provided values.')
    
#     #=== Build coordinates 
    
#     # Primary/Base coordinates
#     coords= {
#         'Time': times,
#         unified_coord_name: unified_dim,
#         }
    
#     # Virtual coordinates (dict-of-dicts / named fields)
#     if dict_of == 'dicts':
#         for vcoord in virtual_coord_names:
#             try: vcoord_values= [virtual_map_dict[unified][vcoord] for unified in unified_dim]
#             except KeyError as e:
#                 raise KeyError(f'Missing key {e} in virtual_map_dict entries (dict_of="dicts")') from e
            
#             coords[vcoord] = (unified_coord_name, vcoord_values)
    
    
#     # Virtual coordinates (dict-of-tuples / positional fields)
#     elif dict_of == 'tuples':
#         for i, vcoord in enumerate(virtual_coord_names):
#             try: vcoord_values = [virtual_map_dict[unified][i] for unified in unified_dim]
#             except IndexError as e:
#                 raise IndexError(f'Index {i} out of range for virtual_map_dict entries (dict_of="tuples")') from e
            
#             coords[vcoord] = (unified_coord_name, vcoord_values)
    
#     else: raise TypeError(f"dict_of parameter must be either 'dicts' or 'tuples', got: {dict_of}")

#     ########################################################
#     """
#     NOTE: virtual_map_dict can be either:
#       (A) dict-of-dicts (recommended):
#           virtual_map_dict[unified] = {"device": ..., "localID": ..., ...}
#           -> use key lookup: virtual_map_dict[unified][vcoord]
    
#       (B) dict-of-tuples/lists (positional):
#           virtual_map_dict[unified] = (device, localID, ...)
#           -> assumes tuple order matches virtual_coord_names
    
#     Example for (B):
#     for i, vcoord in enumerate(virtual_coord_names):
#         vcoord_values = [virtual_map_dict[unified][i] for unified in unified_dim]
#         coords[vcoord] = (unified_coord_name, vcoord_values)
    
#     Dict-of-tuples requires that the order of elements in the tuple matches the order in virtual_coord_names.
#     """
#     ########################################################

#     #=== Create DataArray
#     da= xr.DataArray(
#         data=values,
#         dims= ['Time', unified_coord_name],
#         coords= coords,
#         name= name,
#         attrs= {
#             'description': f'Base DataArray for unified coordinates of type {name}',
#             'source': 'Constructed using construct_base_da function'
#         }
#     )
#     return da
    











# def construct_channel_type_da(values= None, test_help_values= False, channels=None, channels_ref_dict= None, times= None, name='channel_type_da'):
#     '''
#     Construct a Placeholder DataArray for a set of given channels for all timepoints across all devices for a specific register type.
#     -----------------------------------

#     Parameters:
#         values (list or np.ndarray): Values to populate the DataArray. If None, a placeholder array of NaNs will be created.
#         test_help_values (bool): If True, use test helper values instead of the main values.
#         channels (list): List of channel names to include in the DataArray.
#         channels_ref_dict (dict): Dictionary mapping channel names to their reference information.
#         times (list): List of timepoints to include in the DataArray. This should be obtained using get_df_dict_timestamps with return_type='list'.
#         name (str): Name of the DataArray.

#     Returns:
#         A DataArray populated with the specified values, or a placeholder array if no values are provided.
#     '''

#     assert times is not None and channels is not None and channels_ref_dict is not None, "times, channels, and channels_ref_dict must be provided."

#     if values is None and test_help_values is False:
#         print(f'No values provided, and test_help_values is False. Creating zeros placeholder DataArray with {len(times)} timestamps and {len(channels)} channels.')
#         values= np.zeros((len(times), len(channels)),dtype= bool
#                          )
    
#     if values is None and test_help_values is True:
#         print(f'No values provided, but test_help_values is True. Creating test helper DataArray with {len(times)} timestamps and {len(channels)} channels.')
#         value_list= []
#         for time in times:
#             time_values= []
#             for channel in channels:
#                 device, localID = channels_ref_dict[channel]
#                 time_values.append(f'({channel}, {device}, {localID})')
#             value_list.append(time_values)
#         values= np.array(value_list)
#         print(f'Constructed test helper values array with shape: {values.shape}')
#     # if values is not None and test_help_values is True:
#     #     raise ValueError('Cannot provide both values and set test_help_values to True. Choose one or the other.')

#     devices= [channels_ref_dict[channel][0] for channel in channels]
#     localIDs= [channels_ref_dict[channel][1] for channel in channels]

#     da= xr.DataArray(
#         data=values,
#         dims= ['Time', 'channel'],
#         coords= {
#             'Time': times,
#             'channel': channels,
#             'device': ('channel', devices),
#             'localID': ('channel', localIDs)
#         },
#         name= name,

#         attrs= {
#             'description': f'DataArray for channels of type {name}',
#             'source': 'Constructed using construct_channel_type_da function'
#         }
#     )
#     return da

# Note

Using 
```# Virtual coordinates
    for vcoord in virtual_coord_names:
        vcoord_values= [virtual_map_dict[unified][vcoord] for unified in unified_dim]
        coords[vcoord]= (unified_coord_name, vcoord_values)
```

Requires ref_dict to be dict-of-dicts, rather than dict-of-tuples:

```
channel_ref_dict= {
                f'{channel_list[0]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
                f'{channel_list[1]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
                f'{channel_list[2]}':  {'device': f'{device_list[0]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

                f'{channel_list[3]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
                f'{channel_list[4]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
                f'{channel_list[5]}':  {'device': f'{device_list[1]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

                f'{channel_list[6]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
                f'{channel_list[7]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
                f'{channel_list[8]}':  {'device': f'{device_list[2]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

                f'{channel_list[9]}':  {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
                f'{channel_list[10]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
                f'{channel_list[11]}': {'device': f'{device_list[3]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

                f'{channel_list[12]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
                f'{channel_list[13]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
                f'{channel_list[14]}': {'device': f'{device_list[4]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},

                f'{channel_list[15]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][0]}'},
                f'{channel_list[16]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][1]}'},
                f'{channel_list[17]}': {'device': f'{device_list[5]}', 'localID': f'{channel_type_localIDs[type_key][2]}'},
            }
```

If using dict-of-tuples:
```
            channel_ref_dict= {
                f'{channel_list[0]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][0]}'),
                f'{channel_list[1]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][1]}'),
                f'{channel_list[2]}':  (f'{device_list[0]}', f'{channel_type_localIDs[type_key][2]}'),

                f'{channel_list[3]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][0]}'),
                f'{channel_list[4]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][1]}'),
                f'{channel_list[5]}':  (f'{device_list[1]}', f'{channel_type_localIDs[type_key][2]}'),

                f'{channel_list[6]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][0]}'),
                f'{channel_list[7]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][1]}'),
                f'{channel_list[8]}':  (f'{device_list[2]}', f'{channel_type_localIDs[type_key][2]}'),

                f'{channel_list[9]}':  (f'{device_list[3]}', f'{channel_type_localIDs[type_key][0]}'),
                f'{channel_list[10]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][1]}'),
                f'{channel_list[11]}': (f'{device_list[3]}', f'{channel_type_localIDs[type_key][2]}'),

                f'{channel_list[12]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][0]}'),
                f'{channel_list[13]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][1]}'),
                f'{channel_list[14]}': (f'{device_list[4]}', f'{channel_type_localIDs[type_key][2]}'),

                f'{channel_list[15]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][0]}'),
                f'{channel_list[16]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][1]}'),
                f'{channel_list[17]}': (f'{device_list[5]}', f'{channel_type_localIDs[type_key][2]}'),
            }

```

Must instead use:
```
for i, vc_name in enumerate(virtual_coord_names):
    vc_values = [channels_ref_dict[ch][i] for ch in channels]
    coords[vc_name] = ('channel', vc_values)
```